# SW2PLA — Complete Theory Reference
**Praktisk Lineær Algebra for Software Udviklere**

This notebook covers every topic in the curriculum with:
- Plain-English explanations
- LaTeX formulas with full symbol breakdowns
- Simple code examples
- Real-world examples
- Exam-style questions and answers

---

## Table of Contents
1. [Vector Norms](#1)
2. [Unit Vector](#2)
3. [Dot Product & Orthogonality](#3)
4. [Linear Independence & Span](#4)
5. [Basis & Basis Transformation](#5)
6. [Matrices — Fundamentals](#6)
7. [Matrix Norm (Frobenius)](#7)
8. [Matrix Inverse](#8)
9. [Rank & Nullspace](#9)
10. [Linear Maps](#10)
11. [Transformation Matrices](#11)
12. [Covariance Matrix](#12)
13. [LU Decomposition](#13)
14. [QR Decomposition & Orthogonal Matrices](#14)
15. [Solving Linear Systems](#15)
16. [GLM & Least Squares](#16)
17. [Eigendecomposition](#17)
18. [SVD — Singular Value Decomposition](#18)
19. [Low-Rank Approximation](#19)
20. [PCA — Principal Component Analysis](#20)
21. [GED / LDA](#21)
22. [Master Reference Table](#22)


In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
from sympy import Matrix as symMatrix


---
## 1. Vector Norms <a id='1'></a>

### What is it?
A norm is a mathematical measure of the **size** or **length** of a vector.
Think of it as the distance from the origin to the point described by the vector.
There are multiple types of norms — each measuring "length" slightly differently.

### What is it used for?
- Measuring the magnitude of a vector
- Computing unit vectors (normalisation)
- Measuring distances between points
- Checking convergence in algorithms

### Key questions

**What is the difference between L1, L2, and max norm?**
- **L1**: sum of absolute values — penalises all elements equally
- **L2**: square root of sum of squares — the standard "straight-line" distance
- **Max**: the single largest absolute value — ignores everything except the biggest element

**Why do we use L2 norm most often in linear algebra?**
Because it corresponds to the geometric (Euclidean) distance we know intuitively, and it behaves well mathematically (differentiable everywhere).

**What is the norm of the zero vector?**
Always 0 — and it is the only vector with norm 0.

---

### Formulas

$$\|v\|_1 = \sum_{i=1}^n |v_i| \qquad \|v\|_2 = \sqrt{\sum_{i=1}^n v_i^2} \qquad \|v\|_\infty = \max_i |v_i|$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $v$ | The input vector | Given |
| $v_i$ | The $i$-th element of $v$ | Index into the vector |
| $n$ | Number of elements in $v$ | `len(v)` |
| $\|v\|_1$ | L1 norm (Manhattan distance) | `np.linalg.norm(v, ord=1)` |
| $\|v\|_2$ | L2 norm (Euclidean length) | `np.linalg.norm(v)` |
| $\|v\|_\infty$ | Max norm (Chebyshev distance) | `np.linalg.norm(v, ord=np.inf)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
v = np.array([3, -4, 0])

l1   = np.linalg.norm(v, ord=1)
l2   = np.linalg.norm(v)
lmax = np.linalg.norm(v, ord=np.inf)

# Manual L2 (without built-in)
l2_manual = np.sqrt(np.sum(v**2))

print("Vector v:", v)
print(f"L1  norm: {l1}")
print(f"L2  norm: {l2}  (manual: {l2_manual})")
print(f"Max norm: {lmax}")


Vector v: [ 3 -4  0]
L1  norm: 7.0
L2  norm: 5.0  (manual: 5.0)
Max norm: 4.0


In [ ]:
# ── Real-world example: measuring similarity between two data points ─────────
# In machine learning, L2 distance between two feature vectors tells us
# how similar two data points are.

point_A = np.array([2.0, 3.0, 5.0])  # e.g. house: 2 rooms, 3km from centre, 5th floor
point_B = np.array([2.5, 3.1, 4.8])

distance = np.linalg.norm(point_A - point_B)
print(f"Distance between A and B: {distance:.4f}")
print("The smaller the distance, the more similar the two points are.")


Distance between A and B: 0.5477
The smaller the distance, the more similar the two points are.


---
## 2. Unit Vector <a id='2'></a>

### What is it?
A unit vector is a vector with **length exactly 1**, pointing in the same direction as the original vector.
The process of creating it is called **normalisation**.

### What is it used for?
- Representing **direction** without caring about magnitude
- Building orthonormal bases (e.g. in QR decomposition)
- Computing projections

### Key questions

**What is the norm of any unit vector?**
Always exactly 1 — by definition.

**Can you compute a unit vector from the zero vector?**
No. Division by zero is undefined. The zero vector has no direction.

**Why do we normalise vectors?**
Because many algorithms only care about direction, not magnitude. Normalisation prevents larger vectors from dominating simply because they are bigger.

---

### Formula

$$\hat{v} = \frac{v}{\|v\|_2}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $\hat{v}$ | The unit vector (result) | Divide $v$ by its norm |
| $v$ | Original input vector | Given |
| $\|v\|_2$ | L2 norm of $v$ | `np.linalg.norm(v)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
v = np.array([4, 3])

norm_v  = np.linalg.norm(v)
unit_v  = v / norm_v

print(f"v:          {v}")
print(f"norm(v):    {norm_v}")
print(f"unit(v):    {unit_v}")
print(f"Verify — norm of unit vector: {np.linalg.norm(unit_v):.6f}  (should be 1.0)")


v:          [4 3]
norm(v):    5.0
unit(v):    [0.8 0.6]
Verify — norm of unit vector: 1.000000  (should be 1.0)


In [ ]:
# ── Real-world example: direction of movement in a game ──────────────────────
# A player is moving from point A to point B.
# We only want the direction — the speed is handled separately.

player_pos  = np.array([1.0, 2.0])
target_pos  = np.array([5.0, 6.0])

direction   = target_pos - player_pos           # raw direction vector
unit_dir    = direction / np.linalg.norm(direction)  # normalise

speed       = 3.0
velocity    = speed * unit_dir

print(f"Direction vector:  {direction}")
print(f"Unit direction:    {unit_dir}")
print(f"Velocity (speed={speed}): {velocity}")


Direction vector:  [4. 4.]
Unit direction:    [0.70710678 0.70710678]
Velocity (speed=3.0): [2.12132034 2.12132034]


---
## 3. Dot Product & Orthogonality <a id='3'></a>

### What is it?
The dot product (inner product) multiplies two vectors element-by-element and sums the results.
It encodes the **relationship between directions** of two vectors.

Two vectors are **orthogonal** if their dot product is zero — meaning they are perpendicular (90°).

### What is it used for?
- Checking if vectors are orthogonal
- Computing projections
- Measuring similarity between vectors (used in PCA, SVD, ML)
- Verifying that a matrix is orthogonal ($Q^T Q = I$)

### Key questions

**What does a dot product of 0 mean?**
The two vectors are orthogonal — they point in completely unrelated directions.

**What does a large positive dot product mean?**
The vectors point roughly in the same direction.

**What does a large negative dot product mean?**
The vectors point roughly in opposite directions.

**Is $v \cdot w$ the same as $w \cdot v$?**
Yes — the dot product is commutative.

---

### Formula

$$v \cdot w = \sum_{i=1}^n v_i w_i = \|v\| \cdot \|w\| \cdot \cos(\theta)$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $v, w$ | Input vectors (same length) | Given |
| $v_i, w_i$ | $i$-th elements of each vector | Index into vector |
| $n$ | Number of elements | `len(v)` |
| $\theta$ | Angle between $v$ and $w$ | `np.arccos(dot / (norm_v * norm_w))` |
| $\cos(\theta) = 0$ | Vectors are orthogonal | dot product = 0 |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
v = np.array([4, 3])
w = np.array([5, 2])

dot = np.dot(v, w)
print(f"v · w = {dot}")
print(f"Orthogonal: {np.isclose(dot, 0)}")

# Compute angle between v and w
angle_rad = np.arccos(dot / (np.linalg.norm(v) * np.linalg.norm(w)))
print(f"Angle between v and w: {np.degrees(angle_rad):.2f}°")

# Check two orthogonal vectors
a = np.array([1, 0])
b = np.array([0, 1])
print(f"\na · b = {np.dot(a, b)} → orthogonal: {np.isclose(np.dot(a, b), 0)}")


v · w = 26
Orthogonal: False
Angle between v and w: 15.07°

a · b = 0 → orthogonal: True


In [ ]:
# ── Real-world example: recommendation system ────────────────────────────────
# Dot product measures how similar a user's preferences are to a movie's features.
# Higher dot product = better match.

user_prefs  = np.array([0.9, 0.1, 0.8])   # loves action, dislikes romance, likes sci-fi
movie_A     = np.array([0.8, 0.1, 0.9])   # action/sci-fi film
movie_B     = np.array([0.1, 0.9, 0.2])   # romance film

score_A = np.dot(user_prefs, movie_A)
score_B = np.dot(user_prefs, movie_B)

print(f"Match score — Movie A (action/sci-fi): {score_A:.2f}")
print(f"Match score — Movie B (romance):       {score_B:.2f}")
print(f"Recommended: {'Movie A' if score_A > score_B else 'Movie B'}")


Match score — Movie A (action/sci-fi): 1.45
Match score — Movie B (romance):       0.34
Recommended: Movie A


---
## 4. Linear Independence & Span <a id='4'></a>

### What is it?
A set of vectors is **linearly independent** if no vector in the set can be written as a linear combination of the others.
Informally: each vector adds a new "direction" that the others cannot reach.

The **span** of a set of vectors is the set of all possible linear combinations — i.e., all points you can reach.

### What is it used for?
- Determining whether a matrix is invertible
- Understanding the dimension of a vector space
- Checking whether a system of equations has a unique solution

### Key questions

**How do you check linear independence?**
Form a matrix with the vectors as columns. If $\det(A) \neq 0$, the vectors are linearly independent. Equivalently, $\text{rank}(A) = n$ (full column rank).

**What does it mean if vectors are linearly dependent?**
At least one vector is redundant — it lies in the span of the others. The matrix they form has $\det = 0$ and rank $< n$.

**What is the span of two non-parallel vectors in $\mathbb{R}^2$?**
The entire plane $\mathbb{R}^2$ — you can reach any point with the right combination.

**What is the span of two parallel vectors?**
Only a line — they are linearly dependent and add no new direction.

---

### Formula

Vectors $v_1, v_2, \ldots, v_k$ are linearly independent if:

$$c_1 v_1 + c_2 v_2 + \cdots + c_k v_k = 0 \implies c_1 = c_2 = \cdots = c_k = 0$$

Practical test via determinant (for square matrices):

$$\det([v_1 | v_2 | \cdots | v_n]) \neq 0 \iff \text{linearly independent}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $v_1 \ldots v_k$ | The vectors to test | Given |
| $c_1 \ldots c_k$ | Scalar coefficients | Solved for — must all be 0 |
| $\det(A)$ | Determinant of matrix formed by vectors | `np.linalg.det(A)` |
| $\text{rank}(A)$ | Number of linearly independent columns | `np.linalg.matrix_rank(A)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
# Independent vectors
v1 = np.array([1, 2])
v2 = np.array([3, 4])
A  = np.column_stack([v1, v2])

det  = np.linalg.det(A)
rank = np.linalg.matrix_rank(A)

print("--- Linearly INDEPENDENT ---")
print(f"det  = {det:.4f}  (≠ 0 → independent)")
print(f"rank = {rank}       (= n → independent)")

# Dependent vectors (v2 = 2*v1)
w1 = np.array([1, 2])
w2 = np.array([2, 4])   # 2 * w1
B  = np.column_stack([w1, w2])

print("\n--- Linearly DEPENDENT ---")
print(f"det  = {np.linalg.det(B):.4f}  (= 0 → dependent)")
print(f"rank = {np.linalg.matrix_rank(B)}       (< n → dependent)")


--- Linearly INDEPENDENT ---
det  = -2.0000  (≠ 0 → independent)
rank = 2       (= n → independent)

--- Linearly DEPENDENT ---
det  = 0.0000  (= 0 → dependent)
rank = 1       (< n → dependent)


In [ ]:
# ── Real-world example: feature redundancy in data ───────────────────────────
# If two features in a dataset are linearly dependent,
# one carries no extra information — it is redundant.

# Feature matrix: [age, income, 2*income]  ← third column is redundant
data = np.array([[25, 50000, 100000],
                 [30, 60000, 120000],
                 [22, 45000,  90000]], dtype=float)

rank = np.linalg.matrix_rank(data)
print(f"Rank of feature matrix: {rank}")
print(f"Number of features:     {data.shape[1]}")
print(f"Linearly independent features: {rank}  (one feature is redundant)")


Rank of feature matrix: 2
Number of features:     3
Linearly independent features: 2  (one feature is redundant)


---
## 5. Basis & Basis Transformation <a id='5'></a>

### What is it?
A **basis** is a set of linearly independent vectors that **span** a vector space.
Every vector in the space can be written as exactly one linear combination of the basis vectors.

A **basis transformation** is changing the coordinates of a vector from one basis to another.
The vector itself does not change — only the numbers we use to describe it.

### What is it used for?
- Changing coordinate systems
- Simplifying computations by choosing a convenient basis
- The foundation of PCA (projecting to a new basis aligned with data variance)

### Key questions

**What is the standard basis?**
The standard basis in $\mathbb{R}^n$ consists of unit vectors along each axis: $e_1 = [1,0,0,\ldots]$, $e_2 = [0,1,0,\ldots]$, etc.

**How do you find coordinates in a new basis $F$?**
Solve $F \cdot c_F = v_{\text{std}}$ for $c_F$.

**How do you go from basis $B$ to basis $F$?**
1. Convert to standard basis: $v_{\text{std}} = B \cdot c_B$
2. Convert to basis $F$: solve $F \cdot c_F = v_{\text{std}}$

**What does it mean intuitively to change basis?**
It is like measuring the same distance in different units — the real-world value is the same, but the numbers you write down are different depending on your reference frame.

---

### Formulas

From basis $B$ to standard basis:
$$v_{\text{std}} = B \cdot c_B$$

From standard to basis $F$:
$$c_F = F^{-1} \cdot v_{\text{std}}$$

Combined (from $B$ to $F$):
$$c_F = F^{-1} \cdot B \cdot c_B$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $B$ | Matrix whose columns are the basis vectors of $B$ | Given |
| $F$ | Matrix whose columns are the basis vectors of $F$ | Given |
| $c_B$ | Coordinates of the vector in basis $B$ | Given |
| $v_{\text{std}}$ | Coordinates in the standard basis | `B @ c_B` |
| $c_F$ | Coordinates in basis $F$ | `np.linalg.solve(F, v_std)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
B = np.array([[1, 0, 1],
              [0, 1, 1],
              [1, 1, 0]], dtype=float)

F = np.array([[2, 1, 0],
              [1, 2, 1],
              [0, 1, 2]], dtype=float)

c_B = np.array([3, 1, 2], dtype=float)   # coordinates in basis B

# Step 1: to standard basis
v_std = B @ c_B
print("Coordinates in standard basis:", v_std)

# Step 2: to basis F
c_F = np.linalg.solve(F, v_std)
print("Coordinates in basis F:       ", np.round(c_F, 6))

# Verify: F @ c_F should equal v_std
print("Verification F @ c_F:         ", np.round(F @ c_F, 6))


Coordinates in standard basis: [5. 3. 4.]
Coordinates in basis F:        [ 3.25 -1.5   2.75]
Verification F @ c_F:          [5. 3. 4.]


In [ ]:
# ── Real-world example: GPS coordinates vs. local map ───────────────────────
# GPS gives coordinates in one basis (latitude/longitude).
# A local map uses a different basis (metres from local origin).
# The physical location is the same — only the numbers change.

# Standard basis: GPS-like (x=East, y=North) in km
# Local basis: rotated 30° and shifted
theta    = np.radians(30)
B_local  = np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

# Location in local coordinates
c_local = np.array([3.0, 2.0])

# Convert to standard (GPS-like) coordinates
v_gps = B_local @ c_local
print(f"Local coordinates:  {c_local}")
print(f"GPS coordinates:    {np.round(v_gps, 4)}")
print("(Same physical location — different numbers)")


Local coordinates:  [3. 2.]
GPS coordinates:    [1.5981 3.2321]
(Same physical location — different numbers)


---
## 6. Matrices — Fundamentals <a id='6'></a>

### What is it?
A matrix is a rectangular array of numbers arranged in rows and columns.
It represents a **linear transformation** — a function that maps vectors to vectors.

### What is it used for?
- Representing systems of linear equations
- Encoding transformations (rotation, scaling, projection)
- Storing datasets (rows = observations, columns = features)

### Key concepts

**Trace** — sum of diagonal elements. Only defined for square matrices.
$$\text{tr}(A) = \sum_{i=1}^n a_{ii}$$

**Determinant** — a scalar that encodes whether a matrix is invertible.
$$\det(A) \neq 0 \iff A \text{ is invertible} \iff \text{columns are linearly independent}$$

**Transpose** — flip rows and columns: $(A^T)_{ij} = A_{ji}$

**Symmetric matrix** — $A = A^T$ (mirrored along the diagonal)

**Diagonal matrix** — only non-zero values on the diagonal

### Formula breakdown — Trace

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Square matrix | Given |
| $a_{ii}$ | Diagonal element at row $i$, column $i$ | `A[i, i]` |
| $n$ | Size of matrix | `A.shape[0]` |
| $\text{tr}(A)$ | Sum of diagonal | `np.trace(A)` |

### Formula breakdown — Determinant

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Square matrix | Given |
| $\det(A)$ | Scalar — volume scaling factor | `np.linalg.det(A)` |
| $\det(A) = 0$ | Singular (not invertible) | Rank $< n$ |
| $\det(A) \neq 0$ | Invertible (full rank) | Rank $= n$ |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[3, 1, 2],
              [0, 4, 1],
              [2, 1, 3]], dtype=float)

print("Matrix A:")
print(A)
print(f"\nShape:      {A.shape}")
print(f"Trace:      {np.trace(A)}")
print(f"Det:        {np.linalg.det(A):.6f}")
print(f"Symmetric:  {np.allclose(A, A.T)}")
print(f"Transpose:")
print(A.T)

# Diagonal matrix
D = np.diag([5, 3, 2])
print("\nDiagonal matrix D:")
print(D)
print(f"Inverse of diagonal — just invert each diagonal element:")
print(np.diag(1 / np.diag(D)))


Matrix A:
[[3. 1. 2.]
 [0. 4. 1.]
 [2. 1. 3.]]

Shape:      (3, 3)
Trace:      10.0
Det:        19.000000
Symmetric:  False
Transpose:
[[3. 0. 2.]
 [1. 4. 1.]
 [2. 1. 3.]]

Diagonal matrix D:
[[5 0 0]
 [0 3 0]
 [0 0 2]]
Inverse of diagonal — just invert each diagonal element:
[[0.2        0.         0.        ]
 [0.         0.33333333 0.        ]
 [0.         0.         0.5       ]]


In [ ]:
# ── Real-world example: confusion matrix in ML ───────────────────────────────
# A confusion matrix is a square matrix showing classifier performance.
# The trace = number of correct predictions.

confusion = np.array([[50,  3,  2],   # class 0: 50 correct, 3 and 2 wrong
                      [ 4, 45,  1],   # class 1
                      [ 2,  2, 41]])  # class 2

total    = confusion.sum()
correct  = np.trace(confusion)
accuracy = correct / total

print("Confusion matrix:")
print(confusion)
print(f"\nTotal predictions:   {total}")
print(f"Correct (trace):     {correct}")
print(f"Accuracy:            {accuracy:.2%}")


Confusion matrix:
[[50  3  2]
 [ 4 45  1]
 [ 2  2 41]]

Total predictions:   150
Correct (trace):     136
Accuracy:            90.67%


---
## 7. Matrix Norm (Frobenius) <a id='7'></a>

### What is it?
The Frobenius norm extends the concept of vector length to matrices.
It treats the matrix as if it were a long vector and computes the L2 norm of all its elements.

### What is it used for?
- Measuring the "size" of a matrix
- Measuring approximation error (e.g. in low-rank approximation)
- Comparing matrices

### Key questions

**Is the Frobenius norm the same as `np.linalg.norm`?**
Yes — Frobenius is the **default** norm for matrices in NumPy.

**What does a Frobenius norm of 0 mean?**
The matrix is the zero matrix — all elements are 0.

**How is it different from the vector L2 norm?**
It is not — the Frobenius norm is exactly the L2 norm applied to all elements, as if you flattened the matrix into a vector.

---

### Formula

$$\|A\|_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n a_{ij}^2}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Input matrix ($m \times n$) | Given |
| $a_{ij}$ | Element at row $i$, column $j$ | `A[i, j]` |
| $m, n$ | Number of rows and columns | `A.shape` |
| $\|A\|_F$ | Frobenius norm (scalar) | `np.linalg.norm(A)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[1, 2],
              [3, 4]], dtype=float)

norm_numpy  = np.linalg.norm(A)
norm_manual = np.sqrt(np.sum(A**2))

print(f"A:
{A}")
print(f"Frobenius norm (numpy):  {norm_numpy}")
print(f"Frobenius norm (manual): {norm_manual}")


SyntaxError: unterminated f-string literal (detected at line 8) (2920647135.py, line 8)

In [ ]:
# ── Real-world example: measuring compression error ──────────────────────────
# After a low-rank approximation, how much information did we lose?

np.random.seed(42)
A = np.random.randn(5, 5)
U, S, Vt = np.linalg.svd(A)

# Rank-2 approximation
k = 2
Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, S)
A_approx = sum(S[i] * np.outer(U[:,i], Vt[i,:]) for i in range(k))

error = np.linalg.norm(A - A_approx)   # Frobenius norm of the difference
print(f"Original matrix norm:      {np.linalg.norm(A):.4f}")
print(f"Approximation norm:        {np.linalg.norm(A_approx):.4f}")
print(f"Error (Frobenius):         {error:.4f}")
print(f"Relative error:            {error / np.linalg.norm(A):.2%}")


---
## 8. Matrix Inverse <a id='8'></a>

### What is it?
The inverse $A^{-1}$ of a matrix $A$ is the matrix that "undoes" $A$.
Multiplying $A$ by its inverse gives the identity matrix.

### What is it used for?
- Solving systems of equations: $x = A^{-1}b$
- Reversing a transformation
- Deriving the normal equations in GLM: $\beta = (X^TX)^{-1}X^Ty$

### Key questions

**When does an inverse exist?**
Only when $\det(A) \neq 0$ — i.e., $A$ is square with full rank and linearly independent columns.

**What are the four equivalent conditions for invertibility?**
1. $\det(A) \neq 0$
2. $\text{rank}(A) = n$ (full rank)
3. Columns are linearly independent
4. The only solution to $Ax = 0$ is $x = 0$

**Should you use `np.linalg.inv` to solve $Ax = b$?**
No — use `np.linalg.solve` instead. It is faster and more numerically stable.

**What is the inverse of a diagonal matrix?**
Just invert each diagonal element: $D^{-1} = \text{diag}(1/d_1, 1/d_2, \ldots)$

**What is a left-inverse (non-square matrix)?**
For a tall matrix $A$ ($m > n$), the left inverse is $(A^TA)^{-1}A^T$.

---

### Formulas

$$A \cdot A^{-1} = A^{-1} \cdot A = I$$

**Key properties:**
- $(A^{-1})^{-1} = A$
- $(A^T)^{-1} = (A^{-1})^T$
- $(AB)^{-1} = B^{-1}A^{-1}$
- $\det(A^{-1}) = 1/\det(A)$

**Left-inverse (for tall matrices):**
$$A^{+} = (A^T A)^{-1} A^T$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Square, invertible matrix | Given |
| $A^{-1}$ | Inverse of $A$ | `np.linalg.inv(A)` |
| $I$ | Identity matrix ($AA^{-1} = I$) | `np.eye(n)` |
| $A^+$ | Left-inverse (tall matrix) | `np.linalg.inv(A.T @ A) @ A.T` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[2, 1, 0],
              [4, 3, 2],
              [1, 0, 5]], dtype=float)

A_inv = np.linalg.inv(A)
print("A_inv:")
print(np.round(A_inv, 4))

# Verify A @ A_inv = I
print("\nVerification A @ A_inv:")
print(np.round(A @ A_inv, 6))

# Diagonal matrix inverse
D = np.diag([4.0, 2.0, 5.0])
D_inv_manual = np.diag(1 / np.diag(D))
print("\nDiagonal inverse (manual):")
print(D_inv_manual)

# Left-inverse for a tall matrix
A_tall = np.array([[1, 2],
                   [3, 4],
                   [5, 6]], dtype=float)
left_inv = np.linalg.inv(A_tall.T @ A_tall) @ A_tall.T
print("\nLeft-inverse of tall matrix:")
print(np.round(left_inv, 4))
print("Verify left_inv @ A_tall = I:")
print(np.round(left_inv @ A_tall, 6))


In [ ]:
# ── Real-world example: decoding a transformation ────────────────────────────
# A robot applies a rotation + scale transformation.
# Given the output, recover the input using the inverse.

angle = np.radians(45)
T = np.array([[2*np.cos(angle), -2*np.sin(angle)],
              [2*np.sin(angle),  2*np.cos(angle)]])  # scale=2, rotate 45°

original = np.array([3.0, 1.0])
transformed = T @ original

# Recover using inverse
recovered = np.linalg.inv(T) @ transformed

print(f"Original:    {original}")
print(f"Transformed: {np.round(transformed, 4)}")
print(f"Recovered:   {np.round(recovered, 6)}")


---
## 9. Rank & Nullspace <a id='9'></a>

### What is it?
**Rank** is the number of linearly independent columns (or rows) in a matrix.
It tells you the true "dimensionality" of the information in the matrix.

The **nullspace** (kernel) of $A$ is the set of all vectors $v$ such that $Av = 0$.
These are vectors that $A$ "destroys" — they map to zero.

### What is it used for?
- Determining whether a system has a unique solution
- Understanding data redundancy
- Diagnosing degenerate situations in algorithms

### Key questions

**What does rank = number of columns mean?**
The matrix has full column rank — all columns are independent — and $Av = 0$ has only the trivial solution $v = 0$.

**What does it mean that the nullspace has dimension > 0?**
There exist non-zero vectors that $A$ maps to zero. This means some columns are linearly dependent — one or more contain redundant information. The system $Av = 0$ has infinitely many solutions.

**What does it mean that the nullspace has dimension = 0?**
Only the zero vector maps to zero. All columns are independent, and $A$ is injective (one-to-one).

**Can the nullspace have negative dimension?**
No. Dimension is always $\geq 0$. The minimum nullspace is just the zero vector (dimension 0).

**What is the rank-nullity theorem?**
The total number of columns always splits cleanly between rank and nullity.

---

### Formula — Rank-Nullity Theorem

$$\text{cols}(A) = \text{rank}(A) + \text{nullity}(A)$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Input matrix ($m \times n$) | Given |
| $\text{rank}(A)$ | Number of linearly independent columns | `np.linalg.matrix_rank(A)` |
| $\text{nullity}(A)$ | Dimension of the nullspace | `A.shape[1] - rank` |
| $\text{cols}(A)$ | Total number of columns | `A.shape[1]` |
| $Av = 0$ | Test if $v$ is in the nullspace | `np.allclose(A @ v, 0)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

rank    = np.linalg.matrix_rank(A)
nullity = A.shape[1] - rank

print(f"Matrix A ({A.shape[0]}x{A.shape[1]}):")
print(A)
print(f"\nRank:                {rank}")
print(f"Nullity:             {nullity}")
print(f"Rank + Nullity:      {rank + nullity}  (= number of columns)")
print(f"Full column rank:    {rank == A.shape[1]}")

# Test a nullspace vector
v = np.array([1, -2, 1])
Av = A @ v
print(f"\nTest v = {v}")
print(f"A @ v = {np.round(Av, 10)}")
print(f"v in nullspace: {np.allclose(Av, 0)}")


In [ ]:
# ── Real-world example: redundant features in a dataset ──────────────────────
# If salary = 12 * monthly_salary, one column is redundant → nullity > 0

data = np.array([[50000, 4167, 25],
                 [60000, 5000, 30],
                 [45000, 3750, 22],
                 [72000, 6000, 35]], dtype=float)

# Add a redundant column: annual = 12 * monthly
data_redundant = np.column_stack([data, data[:, 1] * 12])

rank_original  = np.linalg.matrix_rank(data)
rank_redundant = np.linalg.matrix_rank(data_redundant)

print(f"Original data rank:           {rank_original} / {data.shape[1]} columns")
print(f"Data with redundant column:   {rank_redundant} / {data_redundant.shape[1]} columns")
print(f"Nullity (redundant):          {data_redundant.shape[1] - rank_redundant}")
print("→ One column carries no new information")


---
## 10. Linear Maps <a id='10'></a>

### What is it?
A linear map (linear transformation) is a function $L: V \to W$ between vector spaces that preserves addition and scalar multiplication:

$$L(v_1 + v_2) = L(v_1) + L(v_2) \qquad L(sv) = sL(v)$$

Every matrix defines a linear map: $L(v) = Av$.

### What is it used for?
- Formalising what matrices *do* to vectors
- Understanding kernel (nullspace) and image (column space)
- The basis of all transformations in the course

### Key questions

**What is the kernel (nullspace) of a linear map?**
All vectors that map to zero: $\ker(L) = \{v : L(v) = 0\}$. Same as the nullspace of the matrix.

**What is the image (range) of a linear map?**
All vectors that can be produced as output: $\text{im}(L) = \{L(v) : v \in V\}$. Same as the column space of the matrix.

**What is the rank of a linear map?**
The dimension of the image — how many independent output directions exist.

**Rank-nullity for linear maps:**
$$\dim(V) = \dim(\ker(L)) + \dim(\text{im}(L))$$

---

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $L$ | The linear map (function) | Represented by matrix $A$ |
| $\ker(L)$ | Kernel / nullspace | `np.linalg.matrix_rank(A)` + nullity |
| $\text{im}(L)$ | Image / column space | Columns of $A$ with pivot positions |
| $\text{rank}(L)$ | Dimension of image | `np.linalg.matrix_rank(A)` |
| $\dim(V)$ | Dimension of input space | `A.shape[1]` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
# A = matrix representation of linear map L: R^3 -> R^2
A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)

rank    = np.linalg.matrix_rank(A)
nullity = A.shape[1] - rank

print(f"Linear map: R^{A.shape[1]} → R^{A.shape[0]}")
print(f"Rank   (dim of image):  {rank}")
print(f"Nullity (dim of kernel): {nullity}")
print(f"Check: rank + nullity = {rank + nullity} = dim(input space) = {A.shape[1]}")

# Apply the map to a vector
v = np.array([1, 0, -1], dtype=float)
Lv = A @ v
print(f"\nL({v}) = {Lv}")

# Is v in the kernel?
print(f"v in kernel: {np.allclose(A @ v, 0)}")


In [ ]:
# ── Real-world example: dimensionality reduction as a linear map ─────────────
# A projection matrix maps R^3 → R^2, discarding one dimension.

# Project onto the xy-plane (discard z)
P = np.array([[1, 0, 0],
              [0, 1, 0]], dtype=float)

points_3d = np.array([[1, 2, 5],
                      [3, 4, 8],
                      [2, 1, 3]], dtype=float)

points_2d = (P @ points_3d.T).T  # apply to each row

print("3D points:")
print(points_3d)
print("\nProjected to 2D:")
print(points_2d)
print(f"\nKernel dimension: {P.shape[1] - np.linalg.matrix_rank(P)} (z-axis is lost)")


---
## 11. Transformation Matrices <a id='11'></a>

### What is it?
Transformation matrices encode geometric operations — rotation, scaling, and translation — as matrix multiplications.
In **homogeneous coordinates**, a 2D point $(x, y)$ is represented as $(x, y, 1)$, which allows translation to be included as a matrix multiplication.

### What is it used for?
- Computer graphics (moving and rotating objects)
- Robotics (joint transformations)
- Computer vision (camera transformations)

### Key questions

**Why do we need homogeneous coordinates for translation?**
Translation cannot be expressed as a regular $2 \times 2$ matrix multiplication — it is not a linear operation. By adding a third "dummy" coordinate (always 1), we can represent translation as a $3 \times 3$ matrix.

**What happens when you multiply multiple transformation matrices?**
You get a combined transformation — the order matters! Rotation then translation ≠ translation then rotation.

**How do you reverse a transformation?**
Use the inverse matrix.

---

### Formulas

**Rotation by angle $\theta$:**
$$R = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

**Translation by $(t_x, t_y)$:**
$$T = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ t_x & t_y & 1 \end{bmatrix}$$

**Scaling by $(s_x, s_y)$:**
$$S = \begin{bmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $\theta$ | Rotation angle (radians) | `np.radians(degrees)` |
| $t_x, t_y$ | Translation distances | Given |
| $s_x, s_y$ | Scale factors per axis | Given |
| $R, T, S$ | Transformation matrices | As above |
| Point in homogeneous coords | $(x, y, 1)$ | Append 1 to 2D point |
| Apply transformation | $P_{new} = P \cdot M$ | Multiply point (row) by matrix |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
# A unit square in homogeneous coordinates
square = np.array([[0, 0, 1],
                   [1, 0, 1],
                   [1, 1, 1],
                   [0, 1, 1],
                   [0, 0, 1]], dtype=float)

# Rotation matrix (45°)
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0,              0,             1]])

# Translation matrix (move +2 right, +1 up)
T = np.array([[1, 0, 0],
              [0, 1, 0],
              [2, 1, 1]])

# Scaling matrix (scale x2)
S = np.array([[2, 0, 0],
              [0, 2, 0],
              [0, 0, 1]])

rotated    = square @ R
translated = square @ T
scaled     = square @ S
combined   = square @ R @ T   # rotate then translate

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, pts, title, col in zip(axes,
    [rotated, translated, scaled, combined],
    ["Rotation 45°", "Translation (2,1)", "Scale ×2", "Rotate then Translate"],
    ['red','blue','green','purple']):
    ax.plot(square[:,0], square[:,1], 'k--', alpha=0.3, label='Original')
    ax.plot(pts[:,0],    pts[:,1],    color=col, label='Result')
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal'); ax.grid(True); ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# ── Real-world example: robot arm joint transformation ───────────────────────
# A robot arm has a joint at the origin.
# The arm segment is 3 units long, initially pointing right.
# We rotate it 60° to reach a new position.

arm_start = np.array([0, 0, 1], dtype=float)
arm_end   = np.array([3, 0, 1], dtype=float)

theta = np.radians(60)
R = np.array([[np.cos(theta), -np.sin(theta), 0],
              [np.sin(theta),  np.cos(theta), 0],
              [0,              0,             1]])

new_end = arm_end @ R
print(f"Arm end (original):  {arm_end[:2]}")
print(f"Arm end (after 60°): {np.round(new_end[:2], 4)}")


---
## 12. Covariance Matrix <a id='12'></a>

### What is it?
The covariance matrix summarises how features in a dataset vary **together**.
The diagonal contains the variance of each feature.
The off-diagonal elements show how pairs of features co-vary.

### What is it used for?
- PCA (eigendecomposition of the covariance matrix)
- GED/LDA (between-class and within-class covariance)
- Understanding correlations in data
- Multivariate statistics

### Key questions

**What does a large off-diagonal value mean?**
The two features are strongly correlated — when one goes up, the other tends to go up (positive) or down (negative).

**What does an off-diagonal value of 0 mean?**
The two features are uncorrelated — they vary independently.

**Why do we mean-center before computing covariance?**
Covariance measures variation *around the mean*. Without centering, the result would be dominated by the mean itself, not the spread.

**Is the covariance matrix always symmetric?**
Yes — because $\text{Cov}(x, y) = \text{Cov}(y, x)$.

**Is the covariance matrix always positive semi-definite?**
Yes — all eigenvalues are $\geq 0$.

---

### Formula

$$C = \frac{X^T X}{n - 1}$$

where $X$ is the **mean-centered** data matrix.

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $X$ | Mean-centered data matrix ($n \times p$) | `X - np.mean(X, axis=0, keepdims=True)` |
| $n$ | Number of observations (rows) | `X.shape[0]` |
| $p$ | Number of features (columns) | `X.shape[1]` |
| $n - 1$ | Bessel's correction (unbiased estimate) | Standard: always use $n-1$ |
| $C$ | Covariance matrix ($p \times p$) | `X.T @ X / (n - 1)` |
| $C_{ii}$ | Variance of feature $i$ | Diagonal elements |
| $C_{ij}$ | Covariance between features $i$ and $j$ | Off-diagonal elements |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
np.random.seed(42)
X_raw = np.array([[2.5, 2.4],
                  [0.5, 0.7],
                  [2.2, 2.9],
                  [1.9, 2.2],
                  [3.1, 3.0]], dtype=float)

# Step 1: Mean-center  (axis=0 = per column/feature)
X = X_raw - np.mean(X_raw, axis=0, keepdims=True)

# Step 2: Covariance matrix
n = X.shape[0]
C = X.T @ X / (n - 1)

print("Mean-centered X:")
print(np.round(X, 4))
print(f"\nCovariance matrix C:")
print(np.round(C, 4))
print(f"\nVariance feature 1: {C[0,0]:.4f}")
print(f"Variance feature 2: {C[1,1]:.4f}")
print(f"Covariance(1,2):    {C[0,1]:.4f}  (positive → correlated)")
print(f"Symmetric: {np.allclose(C, C.T)}")


In [ ]:
# ── Real-world example: stock returns correlation ────────────────────────────
# Are two stocks moving together?

stock_A = np.array([1.2, -0.5, 2.1, 0.8, 1.5, -0.3, 1.8])  # daily returns %
stock_B = np.array([1.0, -0.3, 1.8, 0.7, 1.4, -0.1, 1.6])  # similar stock

data = np.column_stack([stock_A, stock_B])
data_centered = data - np.mean(data, axis=0, keepdims=True)
C = data_centered.T @ data_centered / (len(stock_A) - 1)

# Pearson correlation = C[0,1] / sqrt(C[0,0] * C[1,1])
corr = C[0,1] / np.sqrt(C[0,0] * C[1,1])

print(f"Covariance matrix:")
print(np.round(C, 4))
print(f"\nPearson correlation: {corr:.4f}")
print("Interpretation: close to 1 → stocks move very similarly")


---
## 13. LU Decomposition <a id='13'></a>

### What is it?
LU decomposition factors a matrix $A$ into three matrices: a permutation matrix $P$, a lower triangular matrix $L$, and an upper triangular matrix $U$.

$$A = PLU$$

It is essentially Gaussian elimination written as a matrix factorisation.

### What is it used for?
- Solving linear systems efficiently
- Computing determinants
- Understanding Gaussian elimination

### Key questions

**What is a permutation matrix $P$?**
A matrix that reorders the rows of $A$ (for numerical stability). It is **orthogonal**: $P^T P = I$, so $P^{-1} = P^T$.

**What does lower triangular mean?**
All elements above the diagonal are zero. $L$ always has 1s on the diagonal.

**What does upper triangular mean?**
All elements below the diagonal are zero.

**Why is LU useful for solving multiple systems?**
If you need to solve $Ax = b_1$, $Ax = b_2$, $Ax = b_3$... with the same $A$ but different $b$, you only factorise once and solve each system cheaply.

**What is the determinant in terms of LU?**
$\det(A) = \det(U)$ = product of diagonal elements of $U$ (since $\det(P)=\pm1$, $\det(L)=1$).

---

### Formula

$$A = P \cdot L \cdot U$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Original matrix to decompose | Given |
| $P$ | Permutation matrix (row reordering) | `scipy.linalg.lu(A)` |
| $L$ | Lower triangular matrix (1s on diagonal) | `scipy.linalg.lu(A)` |
| $U$ | Upper triangular matrix | `scipy.linalg.lu(A)` |
| $P^TP = I$ | $P$ is orthogonal | `np.allclose(P @ P.T, np.eye(n))` |
| $\det(A)$ | Product of $U$'s diagonal | `np.prod(np.diag(U))` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)

P, L, U = scipy.linalg.lu(A)

print("P (permutation):"); print(P)
print("\nL (lower triangular):"); print(np.round(L, 4))
print("\nU (upper triangular):"); print(np.round(U, 4))

# Verify P @ L @ U = A
print("\nVerification P @ L @ U:"); print(np.round(P @ L @ U, 6))

# P is orthogonal
print(f"\nP @ P.T = I: {np.allclose(P @ P.T, np.eye(3))}")

# Determinant from U
det_from_U = np.prod(np.diag(U))
det_numpy  = np.linalg.det(A)
print(f"\ndet(A) from U diagonal: {det_from_U:.4f}")
print(f"det(A) from numpy:       {det_numpy:.4f}")


In [ ]:
# ── Real-world example: solving many systems efficiently ─────────────────────
# Same circuit (matrix A), different input voltages (b vectors)
A = np.array([[4, -1, 0],
              [-1, 4, -1],
              [0, -1, 4]], dtype=float)

P, L, U = scipy.linalg.lu(A)

# Solve for three different right-hand sides
for i, b in enumerate([np.array([1, 0, 0]),
                        np.array([0, 1, 0]),
                        np.array([0, 0, 1])]):
    x = np.linalg.solve(A, b)
    print(f"b = {b}  →  x = {np.round(x, 4)}")


---
## 14. QR Decomposition & Orthogonal Matrices <a id='14'></a>

### What is it?
QR decomposition factors a matrix $A$ into an **orthogonal matrix** $Q$ and an **upper triangular matrix** $R$.

An orthogonal matrix has columns that are unit vectors and mutually orthogonal — it represents a pure rotation (or reflection).

### What is it used for?
- Solving least squares problems (more stable than normal equations)
- Computing matrix inverses ($A^{-1} = R^{-1}Q^T$)
- Numerical algorithms (Gram-Schmidt)

### Key questions

**What are the four key properties of an orthogonal matrix $Q$?**
1. $Q^T Q = I$ — transpose equals inverse
2. $Q^{-1} = Q^T$ — inversion is free (just transpose)
3. $\det(Q) = \pm 1$ — preserves volumes
4. Columns are unit vectors, mutually orthogonal

**Is the QR decomposition unique?**
No — the signs of the columns of $Q$ and rows of $R$ can vary.

**Why is $Q^{-1} = Q^T$ useful?**
Inverting is expensive — transposing is free. Any rotation can be undone just by transposing.

**What does $R$ look like?**
Upper triangular — all elements below the diagonal are zero.

---

### Formula

$$A = QR$$

$$Q^T Q = I \quad \Rightarrow \quad Q^{-1} = Q^T$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Original matrix to decompose | Given |
| $Q$ | Orthogonal matrix (rotations/reflections) | `np.linalg.qr(A)` |
| $R$ | Upper triangular matrix | `np.linalg.qr(A)` |
| $Q^TQ = I$ | Verify $Q$ is orthogonal | `np.allclose(Q.T @ Q, np.eye(n))` |
| $\det(Q)$ | Should be $\pm 1$ | `np.linalg.det(Q)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[1, 2, 3],
              [0, 1, 4],
              [5, 6, 0]], dtype=float)

Q, R = np.linalg.qr(A)

print("Q (orthogonal):"); print(np.round(Q, 4))
print("\nR (upper triangular):"); print(np.round(R, 4))

# Verify all four orthogonal matrix properties
print(f"\n1. Q.T @ Q = I:     {np.allclose(Q.T @ Q, np.eye(3))}")
print(f"2. Q_inv = Q.T:     {np.allclose(np.linalg.inv(Q), Q.T)}")
print(f"3. det(Q) = ±1:     {abs(round(np.linalg.det(Q), 6))}")
print(f"4. Q @ R = A:       {np.allclose(Q @ R, A)}")


Q (orthogonal):
[[-0.1961 -0.6052 -0.7715]
 [-0.     -0.7868  0.6172]
 [-0.9806  0.121   0.1543]]

R (upper triangular):
[[-5.099  -6.2757 -0.5883]
 [ 0.     -1.271  -4.9629]
 [ 0.      0.      0.1543]]

1. Q.T @ Q = I:     True
2. Q_inv = Q.T:     True
3. det(Q) = ±1:     1.0
4. Q @ R = A:       True


In [ ]:
# ── Real-world example: stable least squares via QR ──────────────────────────
# Normal equations: beta = (X^T X)^-1 X^T y  ← can be numerically unstable
# QR-based:         R @ beta = Q.T @ y        ← more stable

X = np.array([[1, 1], [1, 2], [1, 3], [1, 4], [1, 5]], dtype=float)
y = np.array([2.1, 4.0, 5.9, 8.1, 9.8], dtype=float)

# Standard lstsq
beta_lstsq, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

# QR method
Q, R = np.linalg.qr(X)
beta_qr = np.linalg.solve(R, Q.T @ y)

print(f"beta via lstsq: {np.round(beta_lstsq, 4)}")
print(f"beta via QR:    {np.round(beta_qr, 4)}")
print("(Should be the same)")


---
## 15. Solving Linear Systems <a id='15'></a>

### What is it?
A linear system is a set of equations of the form $Ax = b$, where $A$ is a matrix of coefficients, $x$ is the unknown vector, and $b$ is the right-hand side.

### What is it used for?
- Finding exact solutions to equations
- Analysing whether solutions exist and how many

### Key questions

**How many solutions can a linear system have?**
Exactly three possibilities:
- **Unique solution**: $\text{rank}(A) = \text{rank}([A|b]) = n$
- **Infinitely many**: $\text{rank}(A) = \text{rank}([A|b]) < n$
- **No solution**: $\text{rank}(A) < \text{rank}([A|b])$

**When should you use `np.linalg.solve` vs `np.linalg.lstsq`?**
- `solve`: square matrix, exactly one solution
- `lstsq`: overdetermined or underdetermined — minimises error

**What is RREF used for?**
Row Reduced Echelon Form reveals the structure of solutions — pivot columns, free variables, and whether a solution exists.

---

### Formulas

$$Ax = b \qquad x = A^{-1}b \quad (\text{if } A \text{ is square and invertible})$$

### Formula breakdown — number of solutions via rank

| Condition | Number of solutions |
|-----------|-------------------|
| $\text{rank}([A\|b]) > \text{rank}(A)$ | 0 (inconsistent) |
| $\text{rank}(A) = \text{rank}([A\|b]) = n$ | 1 (unique) |
| $\text{rank}(A) = \text{rank}([A\|b]) < n$ | ∞ (underdetermined) |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
# Unique solution
A = np.array([[2,  1],
              [-1, 3]], dtype=float)
b = np.array([5, 4], dtype=float)

x = np.linalg.solve(A, b)
print(f"Unique solution: x={x[0]:.4f}, y={x[1]:.4f}")
print(f"Verify A @ x = {np.round(A @ x, 6)} (should be {b})")

# RREF analysis
from sympy import Matrix
Ab = np.hstack([A, b.reshape(-1,1)])
rref, pivots = Matrix(Ab).rref()
print(f"\nRREF of [A|b]:")
print(rref)
print(f"Rank A:    {np.linalg.matrix_rank(A)}")
print(f"Rank [A|b]: {np.linalg.matrix_rank(Ab)}")


In [ ]:
# ── Real-world example: traffic flow analysis ─────────────────────────────────
# At each intersection, flow in = flow out.
# System of equations → solve for unknown flows.

# 3 intersections, 3 unknown flows x1, x2, x3
A = np.array([[ 1, -1,  0],
              [ 0,  1, -1],
              [-1,  0,  1]], dtype=float)
b = np.array([10, 5, -15], dtype=float)

rank_A  = np.linalg.matrix_rank(A)
rank_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))

print(f"Rank A:      {rank_A}")
print(f"Rank [A|b]:  {rank_Ab}")

if rank_A == rank_Ab == A.shape[1]:
    x = np.linalg.solve(A, b)
    print(f"Unique solution: {x}")
elif rank_A == rank_Ab:
    print("Infinitely many solutions")
else:
    print("No solution")


---
## 16. GLM & Least Squares <a id='16'></a>

### What is it?
The Generalised Linear Model (GLM) fits a linear relationship between inputs and outputs when the system is **overdetermined** (more equations than unknowns — more data points than parameters).

Instead of an exact solution, it finds the $\beta$ that **minimises the sum of squared errors**.

### What is it used for?
- Linear regression
- Fitting models to data
- Predicting new values

### Key questions

**What are the 4 steps of GLM?**
1. Define the model equation: $y = \beta_1 x + \beta_0$
2. Map data to equations — build the **design matrix** $X$
3. Write as matrix equation: $y = X\beta$
4. Solve: $\beta = (X^TX)^{-1}X^Ty$ using `lstsq`

**What is SSE?**
Sum of Squared Errors — the total squared distance between actual and predicted values. Lower = better fit.

**What is R²?**
Coefficient of determination — proportion of variance explained by the model. $R^2 = 1$ is a perfect fit, $R^2 = 0$ means the model explains nothing.

**What is the design matrix?**
A matrix where each row is one data point and each column is a feature (plus a column of ones for the intercept).

---

### Formulas

$$y = X\beta \qquad \beta = (X^TX)^{-1}X^Ty$$

$$\text{SSE} = \sum_{i=1}^n (y_i - \hat{y}_i)^2 \qquad R^2 = 1 - \frac{\text{SSE}}{\sum_{i=1}^n(y_i - \bar{y})^2}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $y$ | Observed output values (vector, length $n$) | Given |
| $X$ | Design matrix ($n \times p$) | `np.column_stack([x, np.ones(n)])` |
| $\beta$ | Parameter vector ($\beta_1, \beta_0$) | `np.linalg.lstsq(X, y, rcond=None)[0]` |
| $\hat{y}$ | Predicted values | `X @ beta` |
| $\bar{y}$ | Mean of observed values | `np.mean(y)` |
| $\text{SSE}$ | Sum of squared residuals | `np.sum((y - y_hat)**2)` |
| $\text{SS}_{tot}$ | Total sum of squares | `np.sum((y - np.mean(y))**2)` |
| $R^2$ | Explained variance ratio | `1 - SSE / SS_tot` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
hours  = np.array([3, 5, 6, 8, 9, 4, 7], dtype=float)
scores = np.array([55, 70, 75, 88, 90, 62, 82], dtype=float)

# Design matrix: [x, 1]
X = np.column_stack([hours, np.ones(len(hours))])

# Solve
beta, _, _, _ = np.linalg.lstsq(X, scores, rcond=None)
beta1, beta0  = beta
print(f"Model: y = {beta1:.4f}*x + {beta0:.4f}")

# Predict
y_hat = X @ beta
pred  = beta1 * 6.5 + beta0
print(f"Prediction at x=6.5: {pred:.2f}")

# SSE and R²
SSE    = np.sum((scores - y_hat)**2)
SS_tot = np.sum((scores - np.mean(scores))**2)
R2     = 1 - SSE / SS_tot

print(f"SSE: {SSE:.4f}")
print(f"R²:  {R2:.4f}")


In [ ]:
# ── Real-world example: house price prediction ───────────────────────────────
# Predict house price from size and number of rooms (multi-variable GLM)

size  = np.array([50, 65, 80, 95, 110, 75, 60, 120], dtype=float)   # m²
rooms = np.array([2,  3,  3,  4,   4,  3,  2,   5], dtype=float)
price = np.array([1200, 1600, 1900, 2300, 2600, 1800, 1400, 3000], dtype=float)  # 1000 DKK

# Design matrix: [size, rooms, 1]
X = np.column_stack([size, rooms, np.ones(len(size))])
beta, _, _, _ = np.linalg.lstsq(X, price, rcond=None)

print(f"Model: price = {beta[0]:.2f}*size + {beta[1]:.2f}*rooms + {beta[2]:.2f}")

# Predict a 90m², 3-room apartment
pred = beta[0]*90 + beta[1]*3 + beta[2]
print(f"\nPredicted price (90m², 3 rooms): {pred:.0f} thousand DKK")

y_hat  = X @ beta
R2     = 1 - np.sum((price - y_hat)**2) / np.sum((price - np.mean(price))**2)
print(f"R²: {R2:.4f}")


---
## 17. Eigendecomposition <a id='17'></a>

### What is it?
Eigendecomposition factors a square matrix into its **eigenvalues** and **eigenvectors**.
An eigenvector is a special direction that a matrix only *stretches* — never rotates.
The eigenvalue says by how much it stretches.

### What is it used for?
- Understanding what a transformation does geometrically
- PCA (finding directions of maximum variance)
- Diagonalising matrices (making computation easier)
- Computing matrix powers efficiently

### Key questions

**What is an eigenvector geometrically?**
A direction that the matrix leaves unchanged — it only scales the vector, never rotates it.

**What does a large eigenvalue mean?**
The matrix stretches vectors in that direction a lot — it is an "important" direction.

**What is diagonalisation?**
Rewriting $A = V \Lambda V^{-1}$ where $\Lambda$ is diagonal. This separates each direction's scaling from the rotation.

**What is the spectral theorem?**
For symmetric matrices, eigenvectors are always real and mutually orthogonal, so $V^{-1} = V^T$ and $A = Q\Lambda Q^T$.

**When can't you diagonalise a matrix?**
When it has fewer than $n$ linearly independent eigenvectors. Use SVD instead.

---

### Formula

$$Av = \lambda v \qquad A = V\Lambda V^{-1}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Square matrix to decompose | Given |
| $v$ | Eigenvector (direction matrix doesn't rotate) | `np.linalg.eig(A)[1]` — columns |
| $\lambda$ | Eigenvalue (scaling factor for $v$) | `np.linalg.eig(A)[0]` |
| $V$ | Matrix of eigenvectors as columns | `np.linalg.eig(A)[1]` |
| $\Lambda$ | Diagonal matrix of eigenvalues | `np.diag(eigenvalues)` |
| $V^{-1}$ | Inverse of eigenvector matrix | `np.linalg.inv(V)` |
| Symmetric $A$ | $V^{-1} = V^T$ (orthogonal eigenvectors) | Use `np.linalg.eigh` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
B = np.array([[4, 1, 0],
              [0, 3, 0],
              [0, 0, 2]], dtype=float)

eigenvalues, V = np.linalg.eig(B)
D     = np.diag(eigenvalues)
V_inv = np.linalg.inv(V)

print("Eigenvalues:", np.round(eigenvalues, 4))
print("\nV (eigenvectors as columns):")
print(np.round(V, 4))
print("\nD (diagonal eigenvalue matrix):")
print(np.round(D, 4))

# Reconstruct
B_rec = V @ D @ V_inv
print("\nVerification V @ D @ V_inv ≈ B:")
print(np.round(B_rec, 6))

# Verify: A @ v = lambda * v  for each eigenpair
for i in range(len(eigenvalues)):
    lhs = B @ V[:, i]
    rhs = eigenvalues[i] * V[:, i]
    print(f"Eigenpair {i}: Av={np.round(lhs,4)},  λv={np.round(rhs,4)},  match={np.allclose(lhs,rhs)}")


In [ ]:
# ── Real-world example: Google PageRank (simplified) ─────────────────────────
# PageRank is the dominant eigenvector of the web's link matrix.
# The page with the highest eigenvector component is most important.

# Transition matrix: entry [i,j] = probability of going from page j to page i
M = np.array([[0,   0.5, 0.5, 0  ],
              [0.5, 0,   0,   0.5],
              [0.5, 0.5, 0,   0  ],
              [0,   0,   0.5, 0.5]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(M)

# The dominant eigenvector (eigenvalue closest to 1)
idx = np.argmax(eigenvalues.real)
pagerank = np.abs(eigenvectors[:, idx].real)
pagerank /= pagerank.sum()  # normalise to sum = 1

for i, score in enumerate(pagerank):
    print(f"Page {i+1}: PageRank = {score:.4f}")
print(f"Most important page: {np.argmax(pagerank) + 1}")


---
## 18. SVD — Singular Value Decomposition <a id='18'></a>

### What is it?
SVD is the generalisation of eigendecomposition to **all matrices** — not just square ones.
It decomposes any $m \times n$ matrix into three matrices.

### What is it used for?
- Low-rank approximation and compression
- PCA (equivalent to eigendecomposition of the covariance matrix)
- Solving least squares problems
- Noise reduction

### Key questions

**What are U, Σ, and Vᵀ geometrically?**
- $V^T$: rotate the input
- $\Sigma$: scale along each axis (singular values)
- $U$: rotate the output

**What are the singular values?**
Non-negative numbers on the diagonal of $\Sigma$, ordered from largest to smallest.
They measure the "importance" of each component.

**How is SVD related to eigendecomposition?**
- $A^T A = V \Sigma^2 V^T$ → $V$ = eigenvectors of $A^TA$, $\sigma_i^2$ = eigenvalues of $A^TA$
- $A A^T = U \Sigma^2 U^T$ → $U$ = eigenvectors of $AA^T$

**When is SVD equivalent to eigendecomposition?**
When $A$ is square and symmetric — then $U = V$ and singular values = eigenvalues.

**What is the rank of $A$ in terms of SVD?**
The number of non-zero singular values.

---

### Formula

$$A = U \Sigma V^T$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Input matrix ($m \times n$) | Given |
| $U$ | Left singular vectors ($m \times m$, orthogonal) | `np.linalg.svd(A)[0]` |
| $\Sigma$ | Singular values on diagonal ($m \times n$) | `np.diag(S)` after `np.linalg.svd` |
| $V^T$ | Right singular vectors ($n \times n$, orthogonal) | `np.linalg.svd(A)[2]` |
| $S$ | 1D array of singular values | `np.linalg.svd(A)[1]` |
| $\sigma_i^2$ | Eigenvalues of $A^TA$ | `np.linalg.eigvalsh(A.T @ A)` |
| $\text{rank}(A)$ | Number of non-zero singular values | `np.linalg.matrix_rank(A)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
B = np.array([[4, 0, 0],
              [1, 2, 0],
              [-3, 0, 3]], dtype=float)

U, S, Vt = np.linalg.svd(B)

print("U:"); print(np.round(U, 4))
print("\nSingular values S:", np.round(S, 4))
print("\nVt:"); print(np.round(Vt, 4))

# Reconstruct
Sigma = np.zeros_like(B)
np.fill_diagonal(Sigma, S)
B_rec = U @ Sigma @ Vt
print("\nVerification U @ Sigma @ Vt ≈ B:")
print(np.round(B_rec, 6))

# Verify S² = eigenvalues of B.T @ B
evals = np.sort(np.linalg.eigvalsh(B.T @ B))[::-1]
print(f"\nS²:                  {np.round(S**2, 4)}")
print(f"Eigenvalues of BᵀB:  {np.round(evals, 4)}")
print(f"Are equal:           {np.allclose(np.sort(S**2)[::-1], evals)}")


In [ ]:
# ── Real-world example: image compression ────────────────────────────────────
# Represent a grayscale image as a matrix.
# Keep only the top k singular values to compress it.

np.random.seed(42)
# Simulate a simple 20x20 grayscale "image"
image = np.outer(np.sin(np.linspace(0, np.pi, 20)),
                 np.cos(np.linspace(0, np.pi, 20))) + 0.1 * np.random.randn(20, 20)

U, S, Vt = np.linalg.svd(image)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(image, cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')

for ax, k in zip(axes[1:], [1, 3, 10]):
    approx = sum(S[i] * np.outer(U[:,i], Vt[i,:]) for i in range(k))
    ax.imshow(approx, cmap='gray')
    ax.set_title(f"k={k} ({k/20*100:.0f}% of components)")
    ax.axis('off')

plt.suptitle("SVD Image Compression")
plt.tight_layout()
plt.show()

# Show variance captured
total_var = np.sum(S**2)
for k in [1, 3, 10, 20]:
    captured = np.sum(S[:k]**2) / total_var
    print(f"k={k:2d}: {captured:.1%} of variance captured")


---
## 19. Low-Rank Approximation <a id='19'></a>

### What is it?
Any matrix can be written as a sum of rank-1 matrices (outer products), weighted by the singular values.
A **low-rank approximation** keeps only the $k$ most important terms, discarding the rest.

### What is it used for?
- Image and data compression
- Noise removal (small singular values often correspond to noise)
- Dimensionality reduction
- Efficient storage and computation

### Key questions

**What is a rank-1 matrix?**
The outer product of two vectors: $u_i v_i^T$. It has exactly one linearly independent row and one linearly independent column.

**Why do we keep the largest singular values?**
The singular values are ordered from largest to smallest. Larger singular values capture more of the "energy" (information) in the matrix — the first few terms explain most of the structure.

**What does "rank-$k$ approximation" mean?**
It means the approximated matrix has rank at most $k$ — i.e., only $k$ linearly independent columns.

**What is the approximation error?**
The Frobenius norm of $A - \hat{A}_k$. By the Eckart-Young theorem, this is the **best possible** rank-$k$ approximation.

---

### Formulas

Full reconstruction:
$$A = \sum_{i=1}^{r} \sigma_i \, u_i v_i^T$$

Low-rank approximation ($k < r$):
$$\hat{A}_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^T$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $r$ | Rank of $A$ (number of non-zero $\sigma_i$) | `np.linalg.matrix_rank(A)` |
| $k$ | Number of components to keep ($k < r$) | Chosen by you |
| $\sigma_i$ | $i$-th singular value | `S[i]` from `np.linalg.svd` |
| $u_i$ | $i$-th column of $U$ | `U[:, i]` |
| $v_i^T$ | $i$-th row of $V^T$ | `Vt[i, :]` |
| $u_i v_i^T$ | Outer product — rank-1 matrix | `np.outer(U[:,i], Vt[i,:])` |
| $\hat{A}_k$ | Low-rank approximation | `sum(S[i]*np.outer(U[:,i],Vt[i,:]) for i in range(k))` |
| Error | Frobenius norm of residual | `np.linalg.norm(A - A_approx)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[4, 0, 0],
              [1, 2, 0],
              [-3, 0, 3]], dtype=float)

U, S, Vt = np.linalg.svd(A)

print("Singular values:", np.round(S, 4))
print(f"Total energy (sum of S²): {np.sum(S**2):.4f}\n")

# Build approximations rank k=1,2,3
for k in [1, 2, 3]:
    A_k = sum(S[i] * np.outer(U[:,i], Vt[i,:]) for i in range(k))
    err = np.linalg.norm(A - A_k)
    energy = np.sum(S[:k]**2) / np.sum(S**2)
    print(f"k={k}: error={err:.4f},  energy captured={energy:.1%}")


In [ ]:
# ── Real-world example: removing noise from sensor data ──────────────────────
np.random.seed(7)

# True signal: low-rank structure
true_signal = np.outer(np.linspace(1, 5, 15), np.linspace(1, 3, 10))
noisy_data  = true_signal + 1.5 * np.random.randn(*true_signal.shape)

U, S, Vt = np.linalg.svd(noisy_data)

# Use only k=1 to denoise (signal is rank-1 by construction)
k = 1
denoised = sum(S[i] * np.outer(U[:,i], Vt[i,:]) for i in range(k))

error_noisy    = np.linalg.norm(true_signal - noisy_data)
error_denoised = np.linalg.norm(true_signal - denoised)

print(f"Error before denoising: {error_noisy:.4f}")
print(f"Error after  denoising: {error_denoised:.4f}")
print(f"Improvement: {(1 - error_denoised/error_noisy)*100:.1f}%")


---
## 20. PCA — Principal Component Analysis <a id='20'></a>

### What is it?
PCA finds the directions in a dataset along which the data **varies the most**.
It rotates the coordinate system to align with these directions of maximum variance.

### What is it used for?
- Dimensionality reduction (fewer features, less noise)
- Visualisation of high-dimensional data (reduce to 2D or 3D)
- Noise removal
- Feature extraction before machine learning

### Key questions

**What are the principal components?**
The eigenvectors of the covariance matrix — the directions of maximum variance.
PC1 explains the most variance, PC2 the second most, and so on.
All principal components are **orthogonal** to each other.

**What does explained variance tell you?**
What percentage of the total spread in the data is captured by each component.
If PC1 + PC2 explain 95%, you can reduce to 2D and keep 95% of the information.

**Why do we mean-center first?**
PCA measures variance *around the mean*. Without centering, the first component would just point toward the mean of the data, not the direction of most spread.

**Why use `np.linalg.eigh` instead of `np.linalg.eig` for covariance matrices?**
`eigh` is designed for symmetric matrices — it guarantees real eigenvalues and is faster.

**What is the 5-step procedure?**
1. Mean-center data
2. Compute covariance matrix
3. Eigendecompose the covariance matrix
4. Sort by descending eigenvalue
5. Project data onto top $k$ eigenvectors

---

### Formulas

**Covariance matrix:**
$$C = \frac{X^T X}{n-1}$$

**Eigendecomposition:**
$$C = Q \Lambda Q^T$$

**Projection:**
$$Z = X \cdot Q_k$$

**Explained variance:**
$$\text{var}_i = \frac{\lambda_i}{\sum_j \lambda_j} \times 100\%$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $X$ | Mean-centered data ($n \times p$) | `X - np.mean(X, axis=0, keepdims=True)` |
| $n$ | Number of observations | `X.shape[0]` |
| $p$ | Number of features | `X.shape[1]` |
| $C$ | Covariance matrix ($p \times p$) | `X.T @ X / (n-1)` |
| $Q$ | Eigenvectors of $C$ (principal components) | `np.linalg.eigh(C)[1]` |
| $\Lambda$ | Diagonal of eigenvalues | `np.diag(eigenvalues)` |
| $\lambda_i$ | $i$-th eigenvalue (variance in direction $i$) | `np.linalg.eigh(C)[0]` |
| $Q_k$ | First $k$ eigenvectors (sorted desc.) | `evecs[:, :k]` after sorting |
| $Z$ | Projected data ($n \times k$) | `X @ Q_k` |
| $\text{var}_i$ | Fraction of variance explained by $i$-th PC | `evals[i] / sum(evals)` |


In [ ]:
# ── Simple example — all 5 steps ────────────────────────────────────────────
np.random.seed(42)
X_raw = np.random.randn(100, 4)
X_raw[:, 1] += X_raw[:, 0]   # correlation between features 0 and 1
X_raw[:, 3] += X_raw[:, 2]   # correlation between features 2 and 3

# Step 1: Mean-center (axis=0 = per feature/column)
X = X_raw - np.mean(X_raw, axis=0, keepdims=True)

# Step 2: Covariance matrix
C = X.T @ X / (X.shape[0] - 1)
print("Covariance matrix C:"); print(np.round(C, 3))

# Step 3: Eigendecomposition
evals, evecs = np.linalg.eigh(C)

# Step 4: Sort descending
idx   = np.argsort(evals)[::-1]
evals = evals[idx]
evecs = evecs[:, idx]

# Step 5: Project onto 2 principal components
k = 2
Z = X @ evecs[:, :k]
print(f"\nProjected data shape: {Z.shape}")

# Explained variance
var_exp = evals / np.sum(evals) * 100
for i, v in enumerate(var_exp):
    print(f"PC{i+1}: {v:.1f}%")
print(f"Total (2 PCs): {sum(var_exp[:2]):.1f}%")

# Verify: variance of PC1 = eigenvalue 1
print(f"\nVariance of PC1: {np.var(Z[:,0], ddof=1):.4f}  (should be ≈ {evals[0]:.4f})")
# Verify: PCs are uncorrelated
print(f"Correlation PC1/PC2: {np.corrcoef(Z[:,0], Z[:,1])[0,1]:.6f}  (should be ≈ 0)")


In [ ]:
# ── Real-world example: visualising wine data in 2D ──────────────────────────
np.random.seed(0)
n = 60

# Simulate wine data: 5 chemical features (alcohol, acidity, sugar, tannin, pH)
class_1 = np.random.randn(30, 5) + np.array([1, -1, 0.5,  1, -0.5])
class_2 = np.random.randn(30, 5) + np.array([-1, 1, -0.5, -1, 0.5])
X_wine  = np.vstack([class_1, class_2])
labels  = np.array([0]*30 + [1]*30)

# PCA
X_c = X_wine - np.mean(X_wine, axis=0, keepdims=True)
C   = X_c.T @ X_c / (len(X_c) - 1)
evals, evecs = np.linalg.eigh(C)
idx = np.argsort(evals)[::-1]
evecs = evecs[:, idx]; evals = evals[idx]
Z = X_c @ evecs[:, :2]

plt.figure(figsize=(7, 4))
for label, color, name in [(0,'blue','Wine A'), (1,'red','Wine B')]:
    mask = labels == label
    plt.scatter(Z[mask, 0], Z[mask, 1], c=color, alpha=0.6, label=name)
plt.xlabel(f'PC1 ({evals[0]/sum(evals)*100:.1f}% variance)')
plt.ylabel(f'PC2 ({evals[1]/sum(evals)*100:.1f}% variance)')
plt.title('Wine data projected to 2D via PCA')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()


---
## 21. GED / LDA — Generalised Eigendecomposition <a id='21'></a>

### What is it?
GED (Generalised Eigendecomposition) / LDA (Linear Discriminant Analysis) finds the directions that **best separate two or more classes** — maximising between-class variance while minimising within-class variance.

Where PCA finds directions of maximum *total* variance, LDA finds directions of maximum *discriminative* variance.

### What is it used for?
- Classification and pattern recognition
- Dimensionality reduction that preserves class separability
- Face recognition, speech analysis, medical diagnosis

### Key questions

**What is the between-class covariance ($S_b$ / covB)?**
How spread out the class *means* are from each other. Large $S_b$ = classes are far apart.

**What is the within-class covariance ($S_w$ / covW)?**
How spread out the points are *within* each class. Small $S_w$ = tight clusters.

**What do we want to maximise?**
The ratio $S_b / S_w$ — classes far apart, tight within. The GED finds the directions that maximise this.

**How is GED different from standard eigendecomposition?**
Standard: $Av = \lambda v$ — only one matrix.
GED: $S_b v = \lambda S_w v$ — finds eigenvectors relative to a second matrix.

**What does `scipy.linalg.eig(covB, covW)` do?**
Solves the generalised problem $S_b v = \lambda S_w v$ for the best discriminant directions.

---

### Formula

$$S_w^{-1} S_b \, v = \lambda v \quad \Leftrightarrow \quad S_b v = \lambda S_w v$$

**covB (between-class):**
$$S_b = \sum_{c} \frac{1}{n_c}(X_c - \bar{X}_c)^T(X_c - \bar{X}_c)$$

**covW (within-class):**
$$S_w = \frac{1}{N}(X - \bar{X})^T(X - \bar{X})$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $S_b$ / covB | Between-class covariance | Covariance of each class separately, summed |
| $S_w$ / covW | Within-class covariance | Covariance of all data around global mean |
| $v$ | Discriminant direction (eigenvector) | `scipy.linalg.eig(covB, covW)[1]` |
| $\lambda$ | Discriminant score (eigenvalue) | `scipy.linalg.eig(covB, covW)[0]` |
| $X_c$ | Data points belonging to class $c$ | Boolean masking |
| $\bar{X}_c$ | Mean of class $c$ | `np.mean(X_c, axis=0, keepdims=True)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
np.random.seed(0)
N = 200

class1 = np.random.randn(N, 2) + np.array([ 2, -1])
class2 = np.random.randn(N, 2) + np.array([-2,  1])

# covB: between-class covariance (each class centered on its own mean)
mean1 = np.mean(class1, axis=0, keepdims=True)
mean2 = np.mean(class2, axis=0, keepdims=True)
covB  = ((class1 - mean1).T @ (class1 - mean1) +
         (class2 - mean2).T @ (class2 - mean2)) / N

# covW: within-class covariance (all data centered on global mean)
all_data  = np.vstack([class1, class2])
mean_all  = np.mean(all_data, axis=0, keepdims=True)
covW      = (all_data - mean_all).T @ (all_data - mean_all) / (2 * N)

print("covB (between-class):"); print(np.round(covB, 4))
print("\ncovW (within-class):"); print(np.round(covW, 4))

# GED
evals, evecs = scipy.linalg.eig(covB, covW)
evals = evals.real; evecs = evecs.real
idx   = np.argsort(evals)[::-1]
evals = evals[idx]; evecs = evecs[:, idx]

print(f"\nGED eigenvalues: {np.round(evals, 4)}")
print(f"Best discriminant direction: {np.round(evecs[:, 0], 4)}")


In [ ]:
# ── Real-world example: separating two groups ────────────────────────────────
np.random.seed(1)
N = 100

# Two overlapping classes in 2D
class_A = np.random.randn(N, 2) @ np.array([[2, 1], [0, 1]]) + [3, 2]
class_B = np.random.randn(N, 2) @ np.array([[2, 1], [0, 1]]) + [-1, -1]
all_pts  = np.vstack([class_A, class_B])

# Compute covB and covW
mA = np.mean(class_A, axis=0, keepdims=True)
mB = np.mean(class_B, axis=0, keepdims=True)
covB = ((class_A-mA).T@(class_A-mA) + (class_B-mB).T@(class_B-mB)) / N
m_all = np.mean(all_pts, axis=0, keepdims=True)
covW  = (all_pts - m_all).T @ (all_pts - m_all) / (2*N)

evals, evecs = scipy.linalg.eig(covB, covW)
evals = evals.real; evecs = evecs.real
best_axis = evecs[:, np.argmax(evals)]

# Project onto best discriminant axis
proj_A = class_A @ best_axis
proj_B = class_B @ best_axis

plt.figure(figsize=(10, 3))
plt.subplot(1,2,1)
plt.scatter(class_A[:,0], class_A[:,1], alpha=0.4, label='Class A', c='blue')
plt.scatter(class_B[:,0], class_B[:,1], alpha=0.4, label='Class B', c='red')
plt.title("Original 2D data"); plt.legend(); plt.grid(True)

plt.subplot(1,2,2)
plt.hist(proj_A, bins=20, alpha=0.6, label='Class A', color='blue')
plt.hist(proj_B, bins=20, alpha=0.6, label='Class B', color='red')
plt.title("Projected onto best LDA axis"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


---
## 23. Orthogonal Projection & Complement <a id='23'></a>

### What is it?
The **projection** of $u$ onto $v$ is the component of $u$ that points in the same direction as $v$.
The **orthogonal complement** is what remains — the part of $u$ perpendicular to $v$.

Together they decompose $u$ into two orthogonal parts:
$$u = u_\parallel + u_\perp$$

### What is it used for?
- Gram-Schmidt orthogonalisation (building orthonormal bases)
- Least squares (projecting $y$ onto the column space of $X$)
- Understanding QR decomposition

### Key questions

**What is $u_\parallel$?**
The "shadow" of $u$ onto $v$ — how much of $u$ lies in the direction of $v$.

**What is $u_\perp$?**
The remainder — always perpendicular to $v$. Verify: $u_\perp \cdot v = 0$.

**What if $u$ is already in the direction of $v$?**
Then $u_\perp = 0$ and $u_\parallel = u$.

**What if $u \perp v$?**
Then $u_\parallel = 0$ and $u_\perp = u$.

---

### Formulas

$$u_\parallel = \frac{u \cdot v}{\|v\|^2} \, v \qquad u_\perp = u - u_\parallel$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $u$ | Vector to project | Given |
| $v$ | Vector to project onto | Given |
| $u \cdot v$ | Dot product | `np.dot(u, v)` |
| $\|v\|^2$ | Squared norm of $v$ | `np.dot(v, v)` |
| $u_\parallel$ | Projection of $u$ onto $v$ | `(np.dot(u,v) / np.dot(v,v)) * v` |
| $u_\perp$ | Orthogonal complement | `u - u_parallel` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
u = np.array([4, 3], dtype=float)
v = np.array([2, 0], dtype=float)

u_par  = (np.dot(u, v) / np.dot(v, v)) * v
u_perp = u - u_par

print(f"u         = {u}")
print(f"u_par     = {u_par}   (projection onto v)")
print(f"u_perp    = {u_perp}  (orthogonal complement)")
print(f"Verify orthogonality u_perp · v = {np.dot(u_perp, v):.10f}  (should be 0)")
print(f"Verify reconstruction u_par + u_perp = {u_par + u_perp}  (should be {u})")


In [ ]:
# ── Real-world example: shadow on a wall ─────────────────────────────────────
# A flashlight shines along a wall (direction v).
# An object is at position u. Where does its shadow fall on the wall?

wall_dir = np.array([1, 0, 0], dtype=float)   # wall runs along x-axis
object   = np.array([3, 4, 2], dtype=float)   # 3D object position

shadow = (np.dot(object, wall_dir) / np.dot(wall_dir, wall_dir)) * wall_dir
height = object - shadow   # perpendicular distance from wall

print(f"Object position:    {object}")
print(f"Shadow on wall:     {shadow}")
print(f"Height above wall:  {np.linalg.norm(height):.4f} units")


---
## 24. Angle Between Vectors <a id='24'></a>

### What is it?
The angle $\theta$ between two vectors is derived from the dot product formula.
It measures how "aligned" two vectors are.

### What is it used for?
- Measuring similarity between vectors
- Verifying orthogonality ($\theta = 90°$)
- Understanding projections

### Key questions

**What angle means the vectors are orthogonal?**
$\theta = 90°$ → $\cos(90°) = 0$ → dot product = 0.

**What angle means the vectors point the same direction?**
$\theta = 0°$ → $\cos(0°) = 1$ → dot product = $\|u\|\|v\|$.

**What angle means opposite directions?**
$\theta = 180°$ → $\cos(180°) = -1$ → dot product = $-\|u\|\|v\|$.

---

### Formula

$$\cos(\theta) = \frac{u \cdot v}{\|u\| \cdot \|v\|} \qquad \theta = \arccos\left(\frac{u \cdot v}{\|u\| \cdot \|v\|}\right)$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $u, v$ | Input vectors | Given |
| $u \cdot v$ | Dot product | `np.dot(u, v)` |
| $\|u\|, \|v\|$ | Norms | `np.linalg.norm(u)`, `np.linalg.norm(v)` |
| $\cos(\theta)$ | Cosine similarity | `np.dot(u,v) / (norm_u * norm_v)` |
| $\theta$ | Angle in radians | `np.arccos(cosine)` |
| $\theta°$ | Angle in degrees | `np.degrees(theta)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
u = np.array([1, 0], dtype=float)
v = np.array([1, 1], dtype=float)

cos_theta = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
theta_rad = np.arccos(cos_theta)
theta_deg = np.degrees(theta_rad)

print(f"u = {u}, v = {v}")
print(f"cos(θ) = {cos_theta:.4f}")
print(f"θ = {theta_rad:.4f} rad = {theta_deg:.2f}°")

# Verify: orthogonal vectors → 90°
a = np.array([1, 0])
b = np.array([0, 1])
cos_ab = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"\nOrthogonal vectors: θ = {np.degrees(np.arccos(cos_ab)):.1f}° (should be 90°)")


In [ ]:
# ── Real-world example: document similarity (cosine similarity) ──────────────
# Two documents are represented as word-frequency vectors.
# Small angle = similar documents.

doc1 = np.array([3, 2, 0, 1, 0])  # word frequencies
doc2 = np.array([2, 3, 0, 1, 0])  # similar topic
doc3 = np.array([0, 0, 4, 0, 3])  # different topic

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"doc1 vs doc2 (similar topic): cosine = {cosine_sim(doc1, doc2):.4f}, θ = {np.degrees(np.arccos(cosine_sim(doc1,doc2))):.1f}°")
print(f"doc1 vs doc3 (diff topic):    cosine = {cosine_sim(doc1, doc3):.4f}, θ = {np.degrees(np.arccos(cosine_sim(doc1,doc3))):.1f}°")
print("→ Higher cosine = smaller angle = more similar")


---
## 25. Cauchy-Schwarz & Triangle Inequality <a id='25'></a>

### What is it?
Two fundamental inequalities for norms and inner products.

**Cauchy-Schwarz:** The dot product of two vectors can never exceed the product of their norms.

**Triangle inequality:** The norm of a sum is at most the sum of the norms — "the shortest path between two points is a straight line."

### What are they used for?
- Proving properties of norms
- Bounding expressions in proofs
- The cosine formula is valid precisely because of Cauchy-Schwarz (otherwise arccos would be undefined)

---

### Formulas

**Cauchy-Schwarz:**
$$|u \cdot v| \leq \|u\| \cdot \|v\|$$

**Triangle inequality:**
$$\|u + v\| \leq \|u\| + \|v\|$$

### Formula breakdown

| Symbol | Description | How to verify |
|--------|-------------|---------------|
| $\|u \cdot v\|$ | Absolute value of dot product | `abs(np.dot(u, v))` |
| $\|u\| \cdot \|v\|$ | Product of norms | `np.linalg.norm(u) * np.linalg.norm(v)` |
| $\|u + v\|$ | Norm of sum | `np.linalg.norm(u + v)` |
| $\|u\| + \|v\|$ | Sum of norms | `np.linalg.norm(u) + np.linalg.norm(v)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
u = np.array([1, 5, 7], dtype=float)
v = np.array([2, 5, 9], dtype=float)

# Cauchy-Schwarz: |u·v| ≤ ||u|| * ||v||
lhs_cs = abs(np.dot(u, v))
rhs_cs = np.linalg.norm(u) * np.linalg.norm(v)
print("── Cauchy-Schwarz ──────────────────────────────")
print(f"|u · v|         = {lhs_cs:.4f}")
print(f"||u|| * ||v||   = {rhs_cs:.4f}")
print(f"Holds: {lhs_cs <= rhs_cs + 1e-10}")

# Triangle inequality: ||u+v|| ≤ ||u|| + ||v||
lhs_tri = np.linalg.norm(u + v)
rhs_tri = np.linalg.norm(u) + np.linalg.norm(v)
print("\n── Triangle Inequality ─────────────────────────")
print(f"||u + v||       = {lhs_tri:.4f}")
print(f"||u|| + ||v||   = {rhs_tri:.4f}")
print(f"Holds: {lhs_tri <= rhs_tri + 1e-10}")

# Verify for all 3 norms
for ord_name, ord_val in [("L1", 1), ("L2", 2), ("Linf", np.inf)]:
    lhs = np.linalg.norm(u + v, ord=ord_val)
    rhs = np.linalg.norm(u, ord=ord_val) + np.linalg.norm(v, ord=ord_val)
    print(f"Triangle ({ord_name}): {lhs:.4f} ≤ {rhs:.4f}  → {lhs <= rhs + 1e-10}")


---
## 26. Characteristic Polynomial — Eigenvalues by Hand <a id='26'></a>

### What is it?
The characteristic polynomial is the equation you solve to find eigenvalues.
Setting $\det(A - \lambda I) = 0$ gives a polynomial in $\lambda$ whose roots are the eigenvalues.

### What is it used for?
- Finding eigenvalues analytically (by hand for small matrices)
- Understanding the structure of a matrix
- The Spectral Theorem relies on properties of this polynomial

### Key questions

**What is the characteristic polynomial of a $2\times2$ matrix?**
$\lambda^2 - \text{tr}(A)\lambda + \det(A) = 0$

**How many eigenvalues does an $n\times n$ matrix have?**
Exactly $n$ (counting multiplicity), though some may be complex.

**What does the trace tell you about eigenvalues?**
$\text{tr}(A) = \sum \lambda_i$ — the trace equals the sum of eigenvalues.

**What does the determinant tell you?**
$\det(A) = \prod \lambda_i$ — the determinant equals the product of eigenvalues.

---

### Formula

$$\det(A - \lambda I) = 0$$

**For a $2\times 2$ matrix:**
$$\lambda^2 - \text{tr}(A)\lambda + \det(A) = 0$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $A$ | Square matrix | Given |
| $\lambda$ | Unknown eigenvalue | Solve the polynomial |
| $I$ | Identity matrix | `np.eye(n)` |
| $A - \lambda I$ | Shifted matrix | Each diagonal element reduced by $\lambda$ |
| $\det(A - \lambda I)$ | Characteristic polynomial | Set = 0 and solve |
| $\text{tr}(A)$ | Sum of eigenvalues | `np.trace(A)` |
| $\det(A)$ | Product of eigenvalues | `np.linalg.det(A)` |


In [ ]:
# ── Simple example: 2×2 by hand ─────────────────────────────────────────────
# A = [[-1, 1], [-1, 2]]
# det(A - λI) = det([[-1-λ, 1], [-1, 2-λ]])
#             = (-1-λ)(2-λ) - (1)(-1)
#             = λ² - λ - 2 + 1 = λ² - λ - 1... let's compute

A = np.array([[-1, 1],
              [-1, 2]], dtype=float)

# Characteristic polynomial coefficients: λ² - tr(A)λ + det(A) = 0
trace_A = np.trace(A)
det_A   = np.linalg.det(A)
print(f"tr(A)  = {trace_A}  → sum of eigenvalues")
print(f"det(A) = {det_A:.4f}  → product of eigenvalues")
print(f"Characteristic polynomial: λ² - ({trace_A})λ + ({det_A:.4f}) = 0")

# Solve using numpy (roots of polynomial)
coeffs = [1, -trace_A, det_A]
eigenvalues_hand = np.roots(coeffs)
print(f"\nEigenvalues (from polynomial): {np.round(eigenvalues_hand, 4)}")

# Verify with np.linalg.eig
evals_numpy, _ = np.linalg.eig(A)
print(f"Eigenvalues (numpy):           {np.round(np.sort(evals_numpy), 4)}")

# Verify: tr = sum of evals, det = product of evals
print(f"\nVerify: sum(λ) = {sum(eigenvalues_hand):.4f}  should be tr(A) = {trace_A}")
print(f"Verify: prod(λ) = {eigenvalues_hand[0]*eigenvalues_hand[1]:.4f}  should be det(A) = {det_A:.4f}")


In [ ]:
# ── Real-world example: stability analysis ────────────────────────────────────
# In dynamical systems, eigenvalues determine stability.
# If all eigenvalues have negative real parts → system is stable.

A_stable   = np.array([[-2, 1], [-1, -3]], dtype=float)
A_unstable = np.array([[ 2, 1], [ 1,  3]], dtype=float)

for name, mat in [("Stable", A_stable), ("Unstable", A_unstable)]:
    evals = np.linalg.eig(mat)[0]
    stable = all(e.real < 0 for e in evals)
    print(f"{name} system: eigenvalues = {np.round(evals, 4)}, stable = {stable}")


---
## 27. Eigenvectors by Hand <a id='27'></a>

### What is it?
Once you have eigenvalues $\lambda$, you find the corresponding eigenvector by solving:
$$(A - \lambda I)v = 0$$
This means finding the nullspace of $(A - \lambda I)$.

### Key questions

**Why does $(A - \lambda I)v = 0$ give the eigenvector?**
Because $Av = \lambda v$ rearranges to $Av - \lambda v = 0$ → $(A - \lambda I)v = 0$.

**Is the eigenvector unique?**
No — any scalar multiple of an eigenvector is also an eigenvector. NumPy always returns unit vectors.

**What if $(A - \lambda I)$ has rank 0?**
Then every non-zero vector is an eigenvector (e.g. the identity matrix).

---

### Formula

$$Av = \lambda v \implies (A - \lambda I)v = 0$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $\lambda$ | Known eigenvalue | From characteristic polynomial |
| $A - \lambda I$ | Subtract $\lambda$ from each diagonal element | `A - lambda_val * np.eye(n)` |
| $v$ | Eigenvector (in nullspace of $A - \lambda I$) | `sympy RREF` or `np.linalg.eig` |
| Normalise | Convert to unit eigenvector | `v / np.linalg.norm(v)` |


In [ ]:
# ── Simple example: find eigenvector by hand ────────────────────────────────
from sympy import Matrix

A = np.array([[-1, 1],
              [-1, 2]], dtype=float)

# Eigenvalues from numpy (or from polynomial)
evals, evecs_numpy = np.linalg.eig(A)
print(f"Eigenvalues: {np.round(evals, 4)}")

# For each eigenvalue, find eigenvector by solving (A - λI)v = 0
for lam in evals:
    M = A - lam * np.eye(2)
    rref, pivots = Matrix(M).rref()
    print(f"\nλ = {lam:.4f}")
    print(f"(A - λI) = \n{np.round(M, 4)}")
    print(f"RREF: {rref}")
    # The free variable gives the eigenvector direction
    # For a 2x2 with rank 1: if rref = [[1, -k], [0, 0]], eigenvector = [k, 1]

# Verify using numpy
print("\nNumPy eigenvectors (columns):")
print(np.round(evecs_numpy, 4))
print("\nVerify A @ v = λ * v for first eigenpair:")
v0 = evecs_numpy[:, 0]
print(f"A @ v = {np.round(A @ v0, 4)}")
print(f"λ * v = {np.round(evals[0] * v0, 4)}")


---
## 28. Eigenvalues of $A^{-1}$ <a id='28'></a>

### What is it?
If $v$ is an eigenvector of $A$ with eigenvalue $\lambda$, then $v$ is also an eigenvector of $A^{-1}$, but with eigenvalue $1/\lambda$.

### Why does this hold?
From $Av = \lambda v$, multiply both sides by $A^{-1}$ and $1/\lambda$:
$$v = \lambda A^{-1} v \implies A^{-1}v = \frac{1}{\lambda}v$$

### Key questions

**What condition is required?**
$A$ must be invertible ($\lambda \neq 0$).

**What about eigenvectors of $A^T$?**
Same eigenvalues as $A$, but generally different eigenvectors.

---

### Formula

$$Av = \lambda v \implies A^{-1}v = \frac{1}{\lambda}v$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $v$ | Eigenvector (same for $A$ and $A^{-1}$) | `np.linalg.eig(A)[1]` |
| $\lambda$ | Eigenvalue of $A$ | `np.linalg.eig(A)[0]` |
| $1/\lambda$ | Eigenvalue of $A^{-1}$ | `1 / lambda` |
| $A^{-1}$ | Inverse of $A$ | `np.linalg.inv(A)` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
A = np.array([[4, 1],
              [2, 3]], dtype=float)

evals_A,  evecs_A  = np.linalg.eig(A)
evals_Ai, evecs_Ai = np.linalg.eig(np.linalg.inv(A))

print(f"Eigenvalues of A:     {np.round(np.sort(evals_A), 4)}")
print(f"Eigenvalues of A_inv: {np.round(np.sort(evals_Ai), 4)}")
print(f"1/λ values:           {np.round(np.sort(1/evals_A), 4)}")
print(f"Equal: {np.allclose(np.sort(evals_Ai), np.sort(1/evals_A))}")

# Verify for one eigenpair
v = evecs_A[:, 0]
lam = evals_A[0]
A_inv = np.linalg.inv(A)
print(f"\nFor v = {np.round(v, 4)}, λ = {lam:.4f}:")
print(f"A_inv @ v      = {np.round(A_inv @ v, 4)}")
print(f"(1/λ) * v      = {np.round((1/lam) * v, 4)}")
print(f"Equal: {np.allclose(A_inv @ v, (1/lam) * v)}")


---
## 29. Geometric Interpretation of Eigenvectors <a id='29'></a>

### What is it?
A symmetric matrix transforms a sphere into an ellipsoid.
The **eigenvectors** point along the axes of that ellipsoid.
The **eigenvalues** tell you how much the sphere is stretched along each axis.

### What is it used for?
- Building intuition for PCA (the covariance matrix is symmetric → its eigenvectors are the principal axes of the data cloud)
- Understanding how a matrix transforms space

### Key questions

**Why does a symmetric matrix produce orthogonal eigenvectors?**
Spectral theorem: symmetric matrices always have real eigenvalues and mutually orthogonal eigenvectors.

**What does a large eigenvalue mean geometrically?**
The sphere is stretched a lot in that direction — it is a "principal" direction.

**What does a negative eigenvalue mean?**
The sphere is flipped (reflected) in that direction.


In [ ]:
# ── Sphere → ellipsoid transformation ───────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

# Unit sphere
u_angles = np.linspace(0, 2*np.pi, 40)
v_angles = np.linspace(0, np.pi, 20)
x = np.outer(np.cos(u_angles), np.sin(v_angles))
y = np.outer(np.sin(u_angles), np.sin(v_angles))
z = np.outer(np.ones(40),       np.cos(v_angles))
sphere = np.array([x.ravel(), y.ravel(), z.ravel()])

# Symmetric matrix — stretches and rotates
B = np.array([[3, 1, 0],
              [1, 2, 0],
              [0, 0, 4]], dtype=float)

evals, evecs = np.linalg.eigh(B)
transformed = B @ sphere

fig = plt.figure(figsize=(11, 4))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(x, y, z, alpha=0.3, color='blue')
ax1.set_title("Unit sphere (before)"); ax1.set_box_aspect([1,1,1])

x_t = transformed[0].reshape(40, 20)
y_t = transformed[1].reshape(40, 20)
z_t = transformed[2].reshape(40, 20)
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(x_t, y_t, z_t, alpha=0.3, color='red')
# Plot eigenvectors scaled by eigenvalues
for i in range(3):
    vec = evals[i] * evecs[:, i]
    ax2.quiver(0, 0, 0, vec[0], vec[1], vec[2], color='black', linewidth=2)
ax2.set_title("Ellipsoid after B (eigenvectors = axes)")

plt.tight_layout()
plt.show()

print("Eigenvalues (axis lengths):", np.round(evals, 4))
print("Eigenvectors (axis directions):")
print(np.round(evecs, 4))


---
## 30. QR-Based Inverse & LU-Based Operations <a id='30'></a>

### QR-based inverse
Since $A = QR$, and $Q^{-1} = Q^T$:
$$A^{-1} = (QR)^{-1} = R^{-1}Q^T$$

**Why is this useful?**
$R$ is upper triangular, so $R^{-1}$ is cheaper to compute than a general inverse.
QR-based methods are also more numerically stable.

### LU-based determinant
Since $A = PLU$ and $\det(P) = \pm 1$, $\det(L) = 1$ (ones on diagonal):
$$\det(A) = \det(P) \cdot \prod_i U_{ii}$$

### LU-based inverse
$$A^{-1} = (PLU)^{-1} = U^{-1} L^{-1} P^T$$

### $(AB)^{-1} = B^{-1}A^{-1}$ — reverse order!
The inverse of a product reverses the order of the factors.

---

### Formula breakdown

| Formula | Description | Code |
|---------|-------------|------|
| $A^{-1} = R^{-1}Q^T$ | QR-based inverse | `np.linalg.inv(R) @ Q.T` |
| $\det(A) = \det(P) \prod U_{ii}$ | LU determinant | `np.linalg.det(P) * np.prod(np.diag(U))` |
| $A^{-1} = U^{-1}L^{-1}P^T$ | LU inverse | `np.linalg.inv(U) @ np.linalg.inv(L) @ P.T` |
| $(AB)^{-1} = B^{-1}A^{-1}$ | Reverse order | `inv(B) @ inv(A)` |


In [ ]:
# ── QR-based inverse ─────────────────────────────────────────────────────────
A = np.array([[3, 1, 2],
              [0, 4, 1],
              [2, 1, 3]], dtype=float)

Q, R = np.linalg.qr(A)
A_inv_qr     = np.linalg.inv(R) @ Q.T
A_inv_direct = np.linalg.inv(A)

print("QR-based inverse:")
print(np.round(A_inv_qr, 4))
print(f"Matches direct inverse: {np.allclose(A_inv_qr, A_inv_direct)}")
print(f"Error (norm): {np.linalg.norm(A_inv_qr - A_inv_direct):.2e}")

# ── LU-based determinant ──────────────────────────────────────────────────────
P, L, U = scipy.linalg.lu(A)
det_lu     = np.linalg.det(P) * np.prod(np.diag(U))
det_numpy  = np.linalg.det(A)

print(f"\ndet(A) via LU:    {det_lu:.6f}")
print(f"det(A) via numpy: {det_numpy:.6f}")

# ── LU-based inverse ──────────────────────────────────────────────────────────
A_inv_lu = np.linalg.inv(U) @ np.linalg.inv(L) @ P.T
print(f"\nLU-based inverse matches: {np.allclose(A_inv_lu, A_inv_direct)}")

# ── (AB)⁻¹ = B⁻¹A⁻¹ ─────────────────────────────────────────────────────────
B = np.array([[1, 2, 0],
              [0, 1, 3],
              [1, 0, 1]], dtype=float)

AB_inv        = np.linalg.inv(A @ B)
B_inv_A_inv   = np.linalg.inv(B) @ np.linalg.inv(A)
A_inv_B_inv   = np.linalg.inv(A) @ np.linalg.inv(B)

print(f"\n(AB)⁻¹ = B⁻¹A⁻¹: {np.allclose(AB_inv, B_inv_A_inv)}  ✓")
print(f"(AB)⁻¹ = A⁻¹B⁻¹: {np.allclose(AB_inv, A_inv_B_inv)}  ✗ (wrong order!)")


---
## 31. Extra Matrix Properties <a id='31'></a>

### Frobenius norm via trace
The Frobenius norm can be computed using the trace:
$$\|A\|_F = \sqrt{\text{tr}(A^T A)}$$

### Diagonal matrix: determinant and rank
$$\det(D) = \prod_i d_{ii} \qquad \text{rank}(D) = \text{number of non-zero diagonal elements}$$

### Rank invariance
$$\text{rank}(A) = \text{rank}(A^T) = \text{rank}(A^TA) = \text{rank}(AA^T)$$

### Column space membership test
A vector $b$ is in the column space of $A$ if and only if:
$$\text{rank}([A|b]) = \text{rank}(A)$$

---

### Formula breakdown

| Property | Formula | Code |
|----------|---------|------|
| Frobenius via trace | $\sqrt{\text{tr}(A^TA)}$ | `np.sqrt(np.trace(A.T @ A))` |
| Diagonal determinant | $\prod d_{ii}$ | `np.prod(np.diag(D))` |
| Diagonal rank | Count of non-zero $d_{ii}$ | `np.sum(np.diag(D) != 0)` |
| Rank invariance | rank$(A)$ = rank$(A^TA)$ | `np.linalg.matrix_rank` |
| Column space test | rank$([A|b])$ = rank$(A)$? | `np.hstack([A, b])` then rank |


In [ ]:
# ── Frobenius norm via trace ──────────────────────────────────────────────────
A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)

norm_standard = np.linalg.norm(A)
norm_trace    = np.sqrt(np.trace(A.T @ A))
norm_manual   = np.sqrt(np.sum(A**2))

print(f"Frobenius via np.linalg.norm: {norm_standard:.6f}")
print(f"Frobenius via trace formula:  {norm_trace:.6f}")
print(f"Frobenius via manual sum:     {norm_manual:.6f}")
print(f"All equal: {np.allclose([norm_standard, norm_trace, norm_manual], norm_standard)}")

# ── Diagonal matrix properties ────────────────────────────────────────────────
D = np.diag([3.0, 0.0, 5.0, 2.0])

det_prod  = np.prod(np.diag(D))
det_numpy = np.linalg.det(D)
rank_D    = np.sum(np.diag(D) != 0)

print(f"\nDiagonal D = {np.diag(D)}")
print(f"det via product:  {det_prod:.4f}")
print(f"det via numpy:    {round(det_numpy, 4)}")
print(f"Rank (non-zero):  {rank_D}  (numpy: {np.linalg.matrix_rank(D)})")

# ── Rank invariance ────────────────────────────────────────────────────────────
np.random.seed(42)
A = np.random.randn(4, 3)
print(f"\nRank of A:     {np.linalg.matrix_rank(A)}")
print(f"Rank of A.T:   {np.linalg.matrix_rank(A.T)}")
print(f"Rank of A.T@A: {np.linalg.matrix_rank(A.T @ A)}")
print(f"Rank of A@A.T: {np.linalg.matrix_rank(A @ A.T)}")

# ── Column space membership test ──────────────────────────────────────────────
A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
b_in  = np.array([[3], [7], [11]], dtype=float)   # b = A[:,0] + A[:,1]
b_out = np.array([[1], [0], [0]],  dtype=float)

for b, name in [(b_in, "b_in"), (b_out, "b_out")]:
    rank_A  = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(np.hstack([A, b]))
    print(f"\n{name}: rank(A)={rank_A}, rank([A|b])={rank_Ab} → in column space: {rank_A == rank_Ab}")


---
## 32. Pearson Correlation Coefficient <a id='32'></a>

### What is it?
The Pearson correlation coefficient $\rho$ measures the **linear** relationship between two variables.
It ranges from $-1$ (perfect negative) through $0$ (no correlation) to $+1$ (perfect positive).

### What is it used for?
- Quantifying feature correlation (before PCA or GLM)
- Interpreting covariance matrices (diagonal = variance, off-diagonal = covariance)

### Key questions

**What is the relation to the covariance matrix?**
$\rho_{xy} = C_{xy} / \sqrt{C_{xx} \cdot C_{yy}}$ — normalised covariance.

**What is the relation to the dot product?**
$\rho = \frac{\tilde{x} \cdot \tilde{y}}{\|\tilde{x}\| \cdot \|\tilde{y}\|}$ — the cosine of the angle between mean-centered vectors.

---

### Formula

$$\rho = \frac{\tilde{x} \cdot \tilde{y}}{\|\tilde{x}\| \cdot \|\tilde{y}\|} \qquad \text{where} \quad \tilde{x} = x - \bar{x}$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $x, y$ | Two feature vectors | Given |
| $\bar{x}, \bar{y}$ | Means | `np.mean(x)` |
| $\tilde{x} = x - \bar{x}$ | Mean-centered $x$ | `x - np.mean(x)` |
| $\tilde{x} \cdot \tilde{y}$ | Dot product of centered vectors | `np.dot(x_c, y_c)` |
| $\rho$ | Pearson coefficient $\in [-1, 1]$ | `np.corrcoef(x, y)[0,1]` |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
x = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
y = np.array([3.0, 5.0, 7.0, 9.0, 11.0])  # y ≈ x + 1

# Manual calculation
x_c = x - np.mean(x)
y_c = y - np.mean(y)
rho_manual = np.dot(x_c, y_c) / (np.linalg.norm(x_c) * np.linalg.norm(y_c))

# NumPy built-in
rho_numpy = np.corrcoef(x, y)[0, 1]

print(f"x:            {x}")
print(f"y:            {y}")
print(f"ρ (manual):   {rho_manual:.4f}")
print(f"ρ (numpy):    {rho_numpy:.4f}")
print(f"Interpretation: {'strong positive' if rho_manual > 0.9 else 'moderate' if rho_manual > 0.5 else 'weak'} correlation")

# Example with no correlation
z = np.array([1.0, -2.0, 3.0, -4.0, 5.0])
rho_xz = np.corrcoef(x, z)[0, 1]
print(f"\nρ(x, z) = {rho_xz:.4f}  (near 0 → no correlation)")


In [ ]:
# ── Real-world example: feature correlation in a dataset ─────────────────────
np.random.seed(42)
height = np.random.normal(175, 10, 50)
weight = height * 0.5 + np.random.normal(0, 5, 50)   # correlated with height
age    = np.random.uniform(20, 60, 50)                 # uncorrelated

features = {'height': height, 'weight': weight, 'age': age}
names    = list(features.keys())
data_mat = np.column_stack(list(features.values()))

print("Correlation matrix:")
print(f"{'':10s}", end='')
for n in names: print(f"{n:10s}", end='')
print()
corr_matrix = np.corrcoef(data_mat.T)
for i, n in enumerate(names):
    print(f"{n:10s}", end='')
    for j in range(len(names)):
        print(f"{corr_matrix[i,j]:10.4f}", end='')
    print()


---
## 33. Scree Plot <a id='33'></a>

### What is it?
A scree plot visualises the **explained variance per principal component** (or singular value).
It helps you decide how many components to keep.

### What is it used for?
- Choosing $k$ in PCA and SVD
- Understanding how much information each component contains
- Identifying the "elbow" — the point where adding more components gives diminishing returns

### Key questions

**What does the y-axis show?**
The percentage of total variance explained by each component.

**What is the "elbow"?**
The point where the curve bends sharply — components after the elbow explain very little extra.

**How do you compute explained variance?**
For PCA: $\lambda_i / \sum \lambda_j$ × 100%. For SVD: $\sigma_i^2 / \sum \sigma_j^2$ × 100%.

---

### Formula

$$\text{explained variance}_i = \frac{\lambda_i}{\sum_{j=1}^n \lambda_j} \times 100\%$$

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $\lambda_i$ | $i$-th eigenvalue (sorted descending) | `evals[i]` |
| $\sum \lambda_j$ | Total variance | `np.sum(evals)` |
| Cumulative | Running total | `np.cumsum(var_exp)` |


In [ ]:
# ── Scree plot ────────────────────────────────────────────────────────────────
np.random.seed(7)
X_raw = np.random.randn(100, 8)
# Make first 2 components dominate
X_raw[:, 0] *= 5
X_raw[:, 1] *= 3

# PCA
X = X_raw - np.mean(X_raw, axis=0, keepdims=True)
C = X.T @ X / (X.shape[0] - 1)
evals, _ = np.linalg.eigh(C)
evals = np.sort(evals)[::-1]

var_exp    = evals / np.sum(evals) * 100
cumvar_exp = np.cumsum(var_exp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Individual variance
ax1.bar(range(1, len(var_exp)+1), var_exp, color='steelblue', edgecolor='black')
ax1.plot(range(1, len(var_exp)+1), var_exp, 'ro-', markersize=5)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)")
ax1.set_title("Scree Plot")
ax1.grid(True, alpha=0.3)

# Cumulative variance
ax2.plot(range(1, len(cumvar_exp)+1), cumvar_exp, 'bo-', markersize=6)
ax2.axhline(95, color='red', linestyle='--', label='95% threshold')
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance (%)")
ax2.set_title("Cumulative Explained Variance")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("Explained variance per component:")
for i, (v, cv) in enumerate(zip(var_exp, cumvar_exp)):
    print(f"PC{i+1}: {v:.1f}%  (cumulative: {cv:.1f}%)")


---
## 34. SVD for GLM <a id='34'></a>

### What is it?
The GLM solution $\hat{\beta} = (X^TX)^{-1}X^Ty$ can be computed more stably using SVD:

$$X = U\Sigma V^T \implies \hat{\beta} = V\Sigma^{-1}U^Ty$$

### Why is SVD better?
$(X^TX)^{-1}$ can be numerically unstable when $X$ has near-zero singular values (near-collinear features).
SVD avoids explicitly forming $X^TX$.

### Key questions

**What happens if a singular value is very small (near zero)?**
$\Sigma^{-1}$ blows up — the solution becomes unstable. In practice you threshold small singular values to zero (truncated SVD / pseudoinverse).

**What is the pseudoinverse?**
$X^+ = V\Sigma^+U^T$ where $\Sigma^+$ inverts only the non-zero singular values. This gives the least-norm solution.

---

### Formula

$$\hat{\beta} = V\Sigma^{-1}U^Ty$$

### Formula breakdown

| Symbol | Description | How to compute |
|--------|-------------|----------------|
| $X$ | Design matrix | Given |
| $U, \Sigma, V^T$ | SVD of $X$ | `np.linalg.svd(X, full_matrices=False)` |
| $\Sigma^{-1}$ | Invert diagonal (element-wise) | `np.diag(1/S)` |
| $\hat{\beta}$ | GLM coefficients via SVD | `V @ np.diag(1/S) @ U.T @ y` |
| Pseudoinverse | `np.linalg.pinv(X)` | Handles near-zero singular values |


In [ ]:
# ── Simple example ──────────────────────────────────────────────────────────
hours  = np.array([3, 5, 6, 8, 9, 4, 7], dtype=float)
scores = np.array([55, 70, 75, 88, 90, 62, 82], dtype=float)

X = np.column_stack([hours, np.ones(len(hours))])

# Standard lstsq
beta_lstsq, _, _, _ = np.linalg.lstsq(X, scores, rcond=None)

# SVD method
U, S, Vt = np.linalg.svd(X, full_matrices=False)
V = Vt.T
beta_svd = V @ np.diag(1/S) @ U.T @ scores

print(f"beta via lstsq: {np.round(beta_lstsq, 4)}")
print(f"beta via SVD:   {np.round(beta_svd, 4)}")
print(f"Equal: {np.allclose(beta_lstsq, beta_svd)}")

# Pseudoinverse
beta_pinv = np.linalg.pinv(X) @ scores
print(f"beta via pinv:  {np.round(beta_pinv, 4)}")


In [ ]:
# ── Real-world example: near-collinear features ─────────────────────────────
# Two features are nearly identical → X^T X is near-singular
# lstsq and SVD handle it; direct inverse would fail

np.random.seed(0)
x1 = np.random.randn(50)
x2 = x1 + 0.001 * np.random.randn(50)   # nearly identical to x1
y  = 2*x1 + 3 + np.random.randn(50)*0.1

X = np.column_stack([x1, x2, np.ones(50)])

# Direct normal equations (unstable)
try:
    beta_direct = np.linalg.inv(X.T @ X) @ X.T @ y
    print(f"Direct: {np.round(beta_direct, 2)}")
except np.linalg.LinAlgError:
    print("Direct inverse: FAILED (singular)")

# SVD-based (stable)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
print(f"Singular values: {np.round(S, 4)}  ← note tiny value")
beta_svd = Vt.T @ np.diag(1/S) @ U.T @ y
print(f"SVD:    {np.round(beta_svd, 2)}")

# Pseudoinverse (best for near-singular)
beta_pinv = np.linalg.pinv(X) @ y
print(f"Pinv:   {np.round(beta_pinv, 2)}")


---
## 22. Master Reference Table (Updated) <a id='22'></a>

### Quick Python Reference

| Task | Code | Notes |
|------|------|-------|
| L2 norm of vector | `np.linalg.norm(v)` | Default |
| L1 norm | `np.linalg.norm(v, ord=1)` | |
| Max norm | `np.linalg.norm(v, ord=np.inf)` | |
| Manual L2 | `np.sqrt(np.sum(v**2))` | |
| Frobenius norm | `np.linalg.norm(A)` | Default for matrices |
| Frobenius via trace | `np.sqrt(np.trace(A.T @ A))` | |
| Unit vector | `v / np.linalg.norm(v)` | |
| Dot product | `np.dot(v, w)` | |
| Orthogonality check | `np.isclose(np.dot(v, w), 0)` | |
| Angle between vectors | `np.degrees(np.arccos(np.dot(u,v)/(norm_u*norm_v)))` | |
| Projection of u onto v | `(np.dot(u,v) / np.dot(v,v)) * v` | |
| Orthogonal complement | `u - u_parallel` | |
| Pearson correlation | `np.corrcoef(x, y)[0,1]` | |
| Matrix multiply | `A @ B` | |
| Transpose | `A.T` | |
| Trace | `np.trace(A)` | |
| Determinant | `np.linalg.det(A)` | |
| Diagonal det (manual) | `np.prod(np.diag(D))` | |
| Inverse | `np.linalg.inv(A)` | Square, full rank |
| QR-based inverse | `np.linalg.inv(R) @ Q.T` | |
| LU-based inverse | `np.linalg.inv(U) @ np.linalg.inv(L) @ P.T` | |
| LU-based det | `np.linalg.det(P) * np.prod(np.diag(U))` | |
| (AB)⁻¹ | `np.linalg.inv(B) @ np.linalg.inv(A)` | Reversed! |
| Left-inverse (tall) | `np.linalg.inv(A.T @ A) @ A.T` | |
| Rank | `np.linalg.matrix_rank(A)` | |
| Nullity | `A.shape[1] - np.linalg.matrix_rank(A)` | |
| Symmetric check | `np.allclose(A, A.T)` | |
| Orthogonal check | `np.allclose(Q.T @ Q, np.eye(n))` | |
| Column space test | `rank([A|b]) == rank(A)` | |
| Solve Ax=b | `np.linalg.solve(A, b)` | Square only |
| Least squares | `np.linalg.lstsq(X, y, rcond=None)[0]` | |
| Pseudoinverse | `np.linalg.pinv(X)` | Handles near-singular |
| SVD for GLM | `Vt.T @ np.diag(1/S) @ U.T @ y` | |
| RREF | `sympy.Matrix(Ab).rref()` | |
| LU decomp | `scipy.linalg.lu(A)` → P, L, U | |
| QR decomp | `np.linalg.qr(A)` → Q, R | |
| Eigendecomposition | `np.linalg.eig(A)` → vals, vecs | |
| Eigen (symmetric) | `np.linalg.eigh(A)` → vals, vecs | |
| Characteristic poly roots | `np.roots([1, -np.trace(A), np.linalg.det(A)])` | 2×2 only |
| SVD | `np.linalg.svd(A)` → U, S, Vt | |
| SVD (thin) | `np.linalg.svd(A, full_matrices=False)` | |
| GED | `scipy.linalg.eig(covB, covW)` | |
| Build Sigma from S | `Sigma = np.zeros_like(A); np.fill_diagonal(Sigma, S)` | |
| Low-rank approx | `sum(S[i]*np.outer(U[:,i],Vt[i,:]) for i in range(k))` | |
| Mean-center | `X - np.mean(X, axis=0, keepdims=True)` | Per column! |
| Covariance matrix | `X.T @ X / (n - 1)` | After centering |
| Scree plot values | `evals / np.sum(evals) * 100` | |
| Cumulative variance | `np.cumsum(var_exp)` | |

---

### Theory Answers Quick Reference

| Exam question | Key answer |
|---|---|
| When is SVD = eigendecomposition? | When $A$ is square and symmetric |
| How to find $U$ from eigendecomposition? | Eigendecomp of $AA^T$ → eigenvectors = $U$ |
| What does nullity > 0 mean? | Columns linearly dependent; $Av=0$ has non-trivial solutions |
| What does nullity = 0 mean? | All columns independent; only $v=0$ satisfies $Av=0$ |
| What is low-rank approx? | Keep $k$ largest $\sigma_i$; best rank-$k$ approximation |
| Why useful? | Removes noise; compresses data |
| What are principal components? | Directions of max variance; orthogonal; eigenvectors of $C$ |
| What is changing basis? | Same vector, different coordinate system |
| Why is $Q$ special? | $Q^{-1}=Q^T$ (free inversion); preserves lengths and angles |
| How many solutions to $Ax=b$? | rank$(A)$=rank$([A\|b])$=$n$ → 1; rank$(A)$=rank$([A\|b])$<$n$ → ∞; rank$([A\|b])$>rank$(A)$ → 0 |
| What does covB measure? | Spread between class means |
| What does covW measure? | Spread within each class |
| Eigenvalues of $A^{-1}$? | Same eigenvectors as $A$; eigenvalues = $1/\lambda$ |
| $(AB)^{-1}$ = ? | $B^{-1}A^{-1}$ — reversed order! |
| Characteristic polynomial? | $\det(A - \lambda I) = 0$; for 2×2: $\lambda^2 - \text{tr}(A)\lambda + \det(A) = 0$ |
| $\text{tr}(A)$ vs eigenvalues? | $\text{tr}(A) = \sum \lambda_i$ |
| $\det(A)$ vs eigenvalues? | $\det(A) = \prod \lambda_i$ |
| Rank invariance? | rank$(A)$ = rank$(A^T)$ = rank$(A^TA)$ = rank$(AA^T)$ |
| Cauchy-Schwarz? | $|u \cdot v| \leq \|u\| \cdot \|v\|$ |
| Triangle inequality? | $\|u+v\| \leq \|u\| + \|v\|$ |
